In [ ]:
# Core setup
# This cell installs missing libraries only when they are not already available.

import sys
import subprocess
import importlib.util
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

def ensure_package(import_name, pip_name=None):
    """Import a package if available; otherwise install it in the active kernel."""
    if pip_name is None:
        pip_name = import_name

    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"Package already installed: {import_name}")

# Required for the core notebook
ensure_package("openpyxl")
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("scipy")
ensure_package("sklearn", "scikit-learn")

# Optional, but used by specific sections
ensure_package("networkx")
ensure_package("plotly")
ensure_package("umap", "umap-learn")

# GeoPandas stack for maps
ensure_package("geopandas")
ensure_package("shapely")
ensure_package("pyogrio")

import math
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, TwoSlopeNorm
import matplotlib.patches as mpatches

DATA_FILE = Path("input_file/data-sharing-matrix.xlsx")   # repository-relative

OUTDIR = Path("output_files")
OUTDIR.mkdir(exist_ok=True)

CATEGORY_ORDER = ["A1", "A2", "B", "C", "D", "E"]

CATEGORY_LABELS = [
    "A1 unified legal space",
    "A2 convergent architectures",
    "B standard instruments",
    "C operationally conditional",
    "D restricted in practice",
    "E insufficient certainty",
]

CATEGORY_TO_VALUE = {label: i for i, label in enumerate(CATEGORY_ORDER)}
VALUE_TO_CATEGORY = {v: k for k, v in CATEGORY_TO_VALUE.items()}

# Class palette used on braindatagovernance.org (A1 dark green ... E grey)
PALETTE = {
    "A1": "#1b7f4a",
    "A2": "#8fcf8a",
    "B":  "#f1c232",
    "C":  "#e8842c",
    "D":  "#c0392b",
    "E":  "#6b7280",
}

# Website style shared by all figures below
FG, MUTED, LINE = "#1d2230", "#5a6273", "#c9cfd6"
MAP_BG, MAP_LAND, NOT_CODED = "#f4f7f9", "#d9dee3", "#555b63"
REGIONS = ["Europe", "Africa", "Latin America & Caribbean"]
REGION_COL = {"Europe": "#005981", "Africa": "#e57536", "Latin America & Caribbean": "#a9262c"}
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["IBM Plex Sans", "Helvetica", "Arial", "DejaVu Sans"],
    "text.color": FG, "axes.labelcolor": FG, "xtick.color": FG, "ytick.color": FG,
    "savefig.facecolor": "white", "pdf.fonttype": 42,
})

CMAP = ListedColormap([PALETTE[c] for c in CATEGORY_ORDER])
NORM = BoundaryNorm(np.arange(-0.5, len(CATEGORY_ORDER) + 0.5, 1), CMAP.N)

plt.rcParams["figure.dpi"] = 110


# Cross-border legal interoperability visualizations — annotated exploratory notebook

This notebook provides a structured exploratory analysis of a **directional, categorical, country-by-country legal interoperability matrix** for cross-border biomedical and neurodata transfer. The central object is a matrix in which rows represent **origin/exporting jurisdictions**, columns represent **destination/receiving jurisdictions**, and each cell encodes the legal compatibility class for that directed transfer.

The analytical strategy proceeds from the most complete representation of the data to progressively more compressed summaries. The first figures preserve the full directed matrix. Subsequent figures summarize rows, columns, asymmetries, networks, latent country profiles, regional aggregates, and inequality in legal friction. This progression is deliberate: each section asks what information can be gained by simplifying the matrix, while also making explicit what information is lost by each simplification.

Figures are drawn in the style of the interactive versions on [braindatagovernance.org/toolkit/legal-landscape](https://braindatagovernance.org/toolkit/legal-landscape/): the same class palette throughout, jurisdictions grouped by region, an equirectangular map window on the sampled regions with the United States and Canada marked as not coded at country level, and the network of enabled pathways drawn on three regional rings. The quantities plotted are unchanged.

The compatibility classes are treated as ordered categories for several analyses:

- **A1**: directly enabled, same legal space.
- **A2**: directly enabled through adequacy or high convergence.
- **B**: permissible with standard transfer tools, such as a data transfer agreement.
- **C**: permissible but legally conditional or friction-heavy.
- **D**: restricted or difficult.
- **E**: legally uncertain or not presently supported.

For quantitative summaries, the classes are encoded as an ordinal legal-friction scale from A1 = 0 to E = 5. This encoding should be interpreted as an analytical approximation rather than a claim that the intervals between legal classes are exactly equal. It is useful for visualization, clustering, and summary statistics, but the categorical meaning of each class should remain central to interpretation.

## Setup

This section establishes the shared assumptions and visual grammar used throughout the notebook. The most important decision is to define a single ordered category system and a single color palette, so that all subsequent figures can be compared directly. The path `DATA_FILE` should point to the Excel matrix on your local disk. The helper functions load the matrix, harmonize country labels, convert categories into ordinal values, remove self-transfers safely, and save figures in both PNG and PDF formats.

Self-transfers are removed from most summary analyses because they do not represent cross-border data sharing. In many legal matrices, the diagonal is expected to be A1 by construction. Including it would inflate estimates of direct compatibility and obscure the structure of genuine cross-jurisdiction transfer.

### Interpretation and rationale: exploratory plot

This cell generates a plot or saved visual output that contributes to the exploratory analysis of the cross-border legal interoperability matrix. Interpret the figure in relation to the preceding analyses: each plot either preserves, summarizes, aggregates, or re-expresses the directed country-by-country matrix. When interpreting the result, consider how much information has been compressed from the original matrix and whether the plot is best suited for pairwise, country-level, regional, or global inference.

In [ ]:
# Data loading and reusable helper functions

COUNTRY_NAME_FIXES = {
    # United States variants
    "United States": "United States of America",
    "United States of America": "United States of America",
    "USA": "United States of America",
    "U.S.": "United States of America",
    "US": "United States of America",
    "U.S.A.": "United States of America",

    # United Kingdom variants
    "United Kingdom": "United Kingdom",
    "UK": "United Kingdom",
    "Great Britain": "United Kingdom",

    # North America
    "Canada": "Canada",
    "Mexico": "Mexico",

    # Europe
    "Czech Republic": "Czechia",
    "Czechia": "Czechia",

    # Africa / Natural Earth variants
    "Democratic Republic of the Congo": "Dem. Rep. Congo",
    "DR Congo": "Dem. Rep. Congo",
    "DRC": "Dem. Rep. Congo",
    "Congo, Democratic Republic of the": "Dem. Rep. Congo",
    "Republic of the Congo": "Congo",
    "Congo, Republic of the": "Congo",
    "Côte d’Ivoire": "Côte d'Ivoire",
    "Côte d'Ivoire": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Ivory Coast": "Côte d'Ivoire",
    "Eswatini": "eSwatini",
    "Swaziland": "eSwatini",
    "South Sudan": "S. Sudan",
    "Central African Republic": "Central African Rep.",
    "Equatorial Guinea": "Eq. Guinea",
    "Western Sahara": "W. Sahara",
    "Somaliland": "Somaliland",
    "Tanzania": "Tanzania",
    "United Republic of Tanzania": "Tanzania",

    # Latin America / Caribbean variants often encountered
    "Dominican Republic": "Dominican Rep.",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "The Bahamas": "Bahamas",
    "Bahamas": "Bahamas",

    # Spelling in the source matrix
    "Madagaskar": "Madagascar",
}

WEBSITE_EUROPE = {
    "Germany", "France", "Italy", "Spain", "Netherlands", "Belgium", "Sweden", "Austria", "Finland", "Ireland",
    "Denmark", "Luxembourg", "Czechia", "Portugal", "Lithuania", "Estonia", "Poland", "Greece", "Hungary",
    "Slovakia", "Slovenia", "Romania", "United Kingdom", "Switzerland",
}
WEBSITE_LATAM = {
    "Argentina", "Barbados", "Belize", "Bolivia", "Brazil", "Chile", "Colombia", "Costa Rica", "Cuba", "Dominica",
    "Dominican Rep.", "Dominican Republic", "Ecuador", "El Salvador", "Grenada", "Guatemala", "Guyana", "Honduras",
    "Jamaica", "Mexico", "Nicaragua", "Panama", "Paraguay", "Peru", "Saint Lucia", "Trinidad and Tobago", "Uruguay",
}

def region_of(country):
    """Region of a sampled jurisdiction as used on the website (North America is not in the matrix)."""
    if country in WEBSITE_EUROPE:
        return "Europe"
    if country in WEBSITE_LATAM:
        return "Latin America & Caribbean"
    return "Africa"

def normalize_country_name(x):
    """Normalize one country/jurisdiction name for row-column and map matching."""
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = " ".join(x.split())
    return COUNTRY_NAME_FIXES.get(x, x)

def load_directional_matrix(path):
    """Load and clean the directional compatibility matrix.

    Rows are origin/exporting jurisdictions.
    Columns are destination/receiving jurisdictions.

    Important: name harmonization is applied before row-column alignment.
    This prevents countries such as the United States or Canada from being
    dropped when their row and column labels use slightly different names.
    """
    raw = pd.read_excel(path, index_col=0)

    raw.index = raw.index.map(normalize_country_name)
    raw.columns = raw.columns.map(normalize_country_name)

    # Harmonize occasional miscoding.
    raw = raw.replace({"1A": "A1"})

    # If harmonization created duplicate rows or columns, keep the first.
    raw = raw.loc[~raw.index.duplicated(keep="first")]
    raw = raw.loc[:, ~raw.columns.duplicated(keep="first")]

    rows = set(raw.index)
    cols = set(raw.columns)
    row_only = sorted(rows - cols)
    col_only = sorted(cols - rows)

    print(f"Rows in raw matrix: {len(raw.index)}")
    print(f"Columns in raw matrix: {len(raw.columns)}")
    print(f"Rows not present as columns: {row_only}")
    print(f"Columns not present as rows: {col_only}")

    common = [c for c in raw.index if c in raw.columns]
    df = raw.loc[common, common]

    # Order jurisdictions by region (Europe, Africa, Latin America & Caribbean), keeping the
    # source order inside each region, as on the website.
    order = sorted(df.index, key=lambda x: (REGIONS.index(region_of(x)), list(df.index).index(x)))
    df = df.loc[order, order]

    print(f"Jurisdictions retained after row-column alignment: {len(df.index)}")
    print("North America check after alignment:")
    print([c for c in df.index if any(s in c.lower() for s in ["canada", "united", "america", "mexico", "usa"])])

    return df

def categorical_to_numeric(df):
    """Convert legal categories to ordinal numeric values."""
    return df.replace(CATEGORY_TO_VALUE).apply(pd.to_numeric, errors="coerce")

def remove_self_transfers(df):
    """Return a copy of df with diagonal/self-transfers set to NaN.

    This avoids modifying df.values directly, which can trigger
    `ValueError: underlying array is read-only` in some environments.
    """
    arr = df.to_numpy(copy=True, dtype=object)
    np.fill_diagonal(arr, np.nan)
    return pd.DataFrame(arr, index=df.index.copy(), columns=df.columns.copy())

def mode_or_nan(values):
    counts = pd.Series(values).dropna().value_counts()
    if counts.empty:
        return np.nan
    return counts.idxmax()

def add_category_colorbar(
    fig,
    ax,
    im,
    label="Directional Transfer Compatibility Class"
):
    cbar = fig.colorbar(
        im,
        ax=ax,
        ticks=range(len(CATEGORY_ORDER)),
        fraction=0.07,
        pad=0.08
    )

    cbar.ax.set_yticklabels(CATEGORY_LABELS, fontsize=16)
    cbar.ax.tick_params(labelsize=16)

    # Move the label to the left side
    cbar.ax.yaxis.set_label_position("left")

    cbar.set_label(
        label,
        fontsize=25,
        fontweight="bold",
        labelpad=25,
    )

    return cbar

def class_legend_handles(classes=CATEGORY_ORDER, not_coded=False):
    """Legend patches in the website palette; optionally with the 'not coded' grey."""
    h = [mpatches.Patch(facecolor=PALETTE[c], edgecolor="none", label=CATEGORY_LABELS[CATEGORY_ORDER.index(c)]) for c in classes]
    if not_coded:
        h.append(mpatches.Patch(facecolor=NOT_CODED, edgecolor="none", label="Not coded at country level (United States, Canada)"))
    return h

# Small island jurisdictions drawn as markers on the maps (lon, lat)
SMALL_ISLANDS = {
    "Cabo Verde": (-23.6, 15.1), "Comoros": (43.9, -11.9), "Mauritius": (57.6, -20.3), "Seychelles": (55.5, -4.6),
    "Sao Tome and Principe": (6.7, 0.3), "Barbados": (-59.5, 13.2), "Dominica": (-61.4, 15.4), "Grenada": (-61.7, 12.1),
    "Saint Lucia": (-61.0, 13.9), "Trinidad and Tobago": (-61.2, 10.7),
}
MAP_PHI1 = 25   # standard parallel of the website's equirectangular projection

def website_map(matched_geo, colors, caption, legend_fn, outfile_base):
    """Draw a map in the website style: equirectangular window on the sampled regions, pale land,
    white borders, USA/Canada in dark grey, island jurisdictions as dots.
    matched_geo: GeoDataFrame of sampled jurisdictions with a 'country' column; colors: list of colours."""
    fig, ax = plt.subplots(figsize=(13, 7.4), dpi=300)
    ax.set_facecolor(MAP_BG)
    world.plot(ax=ax, color=MAP_LAND, edgecolor="white", linewidth=0.3, zorder=1)
    matched_geo.plot(ax=ax, color=colors, edgecolor="white", linewidth=0.4, zorder=2)
    world[world["ne_iso_a3"].isin(["USA", "CAN"])].plot(ax=ax, color=NOT_CODED, edgecolor="white", linewidth=0.4, zorder=2)
    col_of = dict(zip(matched_geo["country"], colors))
    for name, (lon, lat) in SMALL_ISLANDS.items():
        if name in col_of:
            ax.plot(lon, lat, "o", ms=5, mfc=col_of[name], mec="white", mew=0.7, zorder=4)
    ax.set_xlim(-128, 60); ax.set_ylim(-58, 72)
    ax.set_aspect(1 / math.cos(math.radians(MAP_PHI1)))
    ax.axis("off")
    legend_fn(ax)
    fig.text(0.125, 0.07, caption, color=MUTED, fontsize=8, va="top")
    savefig(fig, outfile_base)
    plt.show()

def savefig(fig, name):
    fig.savefig(OUTDIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTDIR / f"{name}.pdf", bbox_inches="tight")

df = load_directional_matrix(DATA_FILE)
df_no_self = remove_self_transfers(df)

numeric = categorical_to_numeric(df)
numeric_no_self = categorical_to_numeric(df_no_self)

df.head()


## 1. Directional matrix plot

The first analytical step is to inspect the full directed matrix before applying any statistical reduction. This plot preserves the most information: every country-to-country relationship remains visible, and directionality is retained. The row jurisdiction is the exporter of data, and the column jurisdiction is the receiver.

This figure should be read as the empirical foundation for all later summaries. Blocks of similar color suggest groups of jurisdictions with similar interoperability relationships. Sharp horizontal patterns indicate countries with broadly permissive or restrictive **export** profiles. Sharp vertical patterns indicate countries that are broadly easy or difficult to send data **to**. Because the matrix is directional, the upper and lower triangles need not match.

The purpose of this section is not only visualization. It establishes the baseline structure against which all subsequent summaries should be checked. If later compressed plots appear surprising, the matrix is the figure to return to for validation.

### Interpretation and rationale: directional compatibility matrix

This plot is the central representation of the dataset because it preserves the full directed country-by-country structure. Each row is an origin or exporting jurisdiction and each column is a destination or receiving jurisdiction. The color of each cell represents the legal compatibility class for that specific directed transfer. This means that the cell in row A and column B should be read as the regulatory status of a data transfer from A to B, not as a general relationship between A and B.

This matrix is the least compressed visualization in the notebook. It retains the individual pairwise values and therefore serves as the reference point for interpreting all later summaries. The later stacked plots, maps, embeddings, entropy analyses, and network analyses all reduce this matrix in different ways. When those summaries appear surprising, this matrix should be used to identify the specific country pairs responsible for the pattern.

The matrix is drawn as on braindatagovernance.org: jurisdictions are grouped by region (Europe, Africa, Latin America and the Caribbean) in the order of the source table, the diagonal is left blank because self-transfers are treated as missing, and the six classes use the website palette.

Methodologically, the categorical legal labels are mapped to ordinal values only for visualization. A1 is treated as the lowest-friction class and E as the highest-friction class. The colors should therefore be interpreted as ordered legal difficulty, but the original categories remain categorical legal judgments rather than continuous measurements.

In [ ]:
# Directional compatibility matrix, website style: jurisdictions grouped by region,
# white hairlines between cells, diagonal (self-transfer) left blank, region labels coloured.
countries = list(df_no_self.index)
n = len(countries)

img = np.ones((n, n, 3))
for i, a in enumerate(countries):
    for j, b in enumerate(countries):
        v = df_no_self.iloc[i, j]
        if pd.notna(v):
            img[i, j] = matplotlib.colors.to_rgb(PALETTE[v])

fig, ax = plt.subplots(figsize=(12.5, 12.5), dpi=300)
ax.imshow(img, interpolation="none")

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(countries, rotation=90, fontsize=5.6)
ax.set_yticklabels(countries, fontsize=5.6)
ax.xaxis.set_ticks_position("top")
ax.tick_params(length=0, pad=2)
for lab in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
    lab.set_color(REGION_COL[region_of(lab.get_text())])

# cell hairlines and region separators
ax.set_xticks(np.arange(-.5, n, 1), minor=True); ax.set_yticks(np.arange(-.5, n, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=0.4); ax.tick_params(which="minor", length=0)
bounds = np.cumsum([sum(1 for c in countries if region_of(c) == r) for r in REGIONS])[:-1]
for b in bounds:
    ax.axhline(b - .5, color=FG, lw=1.2); ax.axvline(b - .5, color=FG, lw=1.2)
for s in ax.spines.values():
    s.set_edgecolor(LINE)

ax.set_xlabel("Destination (receiving jurisdiction)", color=MUTED, labelpad=8)
ax.xaxis.set_label_position("top")
ax.set_ylabel("Origin (exporting jurisdiction)", color=MUTED, labelpad=8)
ax.legend(handles=class_legend_handles(), loc="upper left", bbox_to_anchor=(0, -0.035), ncol=3,
          frameon=False, fontsize=8, handlelength=1.2, columnspacing=1.6)

savefig(fig, "01_directional_matrix_custom_palette")
plt.show()


### Interpretation of the directional matrix

The matrix should be interpreted as the least compressed representation of the dataset. A visually coherent block indicates that many jurisdictions share similar legal compatibility relationships with one another. A heterogeneous row indicates that the exporting jurisdiction faces different legal conditions depending on the destination. A heterogeneous column indicates that the receiving jurisdiction is treated differently depending on the exporter.

Because the plot uses a categorical palette, adjacent colors should not be interpreted as continuous physical gradients. Instead, the colors encode ordered legal states. The primary interpretive questions are: whether compatibility is clustered by geography or legal family, whether certain jurisdictions appear as broad interoperability hubs, and whether the direction of transfer produces visually detectable asymmetry. The following sections progressively quantify these patterns.

## 2. Per-country stacked distribution plot

Continuing from the full matrix, this section compresses each row into a distribution over legal classes. Instead of asking which class applies to every specific destination, the plot asks: **what is the overall export profile of each jurisdiction?**

For each country, the row of the matrix is converted into proportions across A1, A2, B, C, D, and E after removing self-transfers. The resulting horizontal stacked bar is a summary of how that country behaves as an exporting jurisdiction across all destination countries. This is a row-wise composition analysis.

This plot is useful because it makes country-level heterogeneity more readable than the full matrix. However, it also discards the identity of the destination countries. Two countries can have identical stacked profiles while interacting with different specific partners.

### Interpretation and rationale: per-country outgoing distribution

After inspecting the complete pairwise matrix, the next question is how each country behaves overall as an exporter. This stacked horizontal bar plot compresses each row of the matrix into a distribution over legal classes. Each bar corresponds to one origin jurisdiction, and each colored segment gives the proportion of that country’s outgoing relationships falling into A1, A2, B, C, D, or E.

This plot is useful because it preserves more information than a single average or mode while remaining easier to compare across countries than the full matrix. A country with a large single-color segment has a relatively uniform export profile; a country with multiple substantial segments has a more heterogeneous export profile. Self-transfers are excluded because they are not cross-border transfers and would artificially inflate highly compatible categories.

This analysis should be read as the first country-level reduction of the matrix. It moves from pairwise values to country profiles while retaining the distributional structure of each country’s outgoing relationships.

In [ ]:
rows = []
for country in df_no_self.index:
    values = pd.Series(df_no_self.loc[country].values).dropna()
    freq = values.value_counts(normalize=True).reindex(CATEGORY_ORDER).fillna(0)
    rows.append(freq)

outgoing_dist = pd.DataFrame(rows, index=df_no_self.index)
outgoing_dist = outgoing_dist.sort_values(by="A1", ascending=False)

countries_sorted = outgoing_dist.index.tolist()
y_pos = np.arange(len(countries_sorted))

fig_height = max(10, len(countries_sorted) * 0.8)
fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)

left = np.zeros(len(countries_sorted))
for category in CATEGORY_ORDER:
    vals = outgoing_dist[category].values
    ax.barh(y_pos, vals, left=left, height=0.6, color=PALETTE[category], label=category)
    left += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(countries_sorted, fontsize=10.5)
ax.set_xlabel("Proportion of outgoing transfers", fontsize=11)
ax.set_ylabel("Country / origin jurisdiction", fontsize=11)
ax.set_title("Per-country distribution of outgoing transfer compatibility classes", fontsize=14, pad=14)
ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
savefig(fig, "02_per_country_outgoing_distribution")
plt.show()


### Interpretation of the per-country outgoing distribution

Countries with bars dominated by A1, A2, or B have comparatively permissive or operationally feasible export profiles. Countries with large C, D, or E components face greater conditionality, restriction, or uncertainty when exporting data. The ordering by A1 proportion is mainly a readability device; it places more directly enabled exporters toward one end of the plot.

This figure should be interpreted together with the directional matrix. The stacked plot tells us how much legal friction characterizes each origin country overall, but the matrix identifies which bilateral relationships produce that friction. Thus, the stacked plot is a country-level summary, whereas the matrix remains the relationship-level evidence.

## 3. Import and export mode world maps

The previous section summarized each country by the full distribution of outgoing classes. This section compresses even further: it assigns each country a single modal class. The export mode is the most frequent class across a country’s row, and the import mode is the most frequent class across a country’s column.

The export map answers: **what is the dominant legal compatibility class when this country exports data to others?** The import map answers: **what is the dominant legal compatibility class when other countries send data to this country?**

This is a highly communicative geographic summary, but it is also a strong compression. The mode does not distinguish between a country that is 90% A1 and a country that is 35% A1, 33% B, and 32% C. Therefore, mode maps should be used as overview figures and interpreted alongside the stacked distributions and entropy measures.

In [ ]:
# Robust map setup using controlled ISO-3 matching and 50m Natural Earth boundaries
# Replace the current map setup / mode_df / merge cell with this entire cell.

import sys
import subprocess
import importlib.util
from pathlib import Path
import zipfile
import urllib.request

def ensure_package(import_name, pip_name=None):
    if pip_name is None:
        pip_name = import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

ensure_package("geopandas")
ensure_package("pycountry")

import geopandas as gpd
import pycountry

NATURAL_EARTH_URL = (
    "https://naturalearth.s3.amazonaws.com/"
    "50m_cultural/ne_50m_admin_0_countries.zip"
)

MAP_DIR = Path("map_data_50m")
MAP_DIR.mkdir(exist_ok=True)

ZIP_FILE = MAP_DIR / "ne_50m_admin_0_countries.zip"
WORLD_FILE = MAP_DIR / "ne_50m_admin_0_countries.shp"

if not WORLD_FILE.exists():
    print("Downloading Natural Earth 50m country boundaries...")
    urllib.request.urlretrieve(NATURAL_EARTH_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extractall(MAP_DIR)

world = gpd.read_file(WORLD_FILE)

iso_candidates = ["ADM0_A3", "ISO_A3", "SOV_A3"]
iso_col = next((c for c in iso_candidates if c in world.columns), None)

if iso_col is None:
    raise ValueError(
        f"No ISO-3 country code column found. Available columns: {list(world.columns)}"
    )

print(f"Using Natural Earth ISO column: {iso_col}")

COUNTRY_TO_ISO3 = {
    # North America
    "Canada": "CAN",
    "Mexico": "MEX",
    "United States": "USA",
    "United States of America": "USA",
    "USA": "USA",
    "US": "USA",
    "U.S.": "USA",
    "U.S.A.": "USA",

    # Europe / UK
    "United Kingdom": "GBR",
    "UK": "GBR",
    "Great Britain": "GBR",
    "Czechia": "CZE",
    "Czech Republic": "CZE",

    # Africa
    "Algeria": "DZA",
    "Angola": "AGO",
    "Benin": "BEN",
    "Botswana": "BWA",
    "Burkina Faso": "BFA",
    "Cabo Verde": "CPV",
    "Cape Verde": "CPV",
    "Cameroon": "CMR",
    "Comoros": "COM",
    "Republic of the Congo": "COG",
    "Congo, Republic of the": "COG",
    "Congo": "COG",
    "Democratic Republic of the Congo": "COD",
    "Dem. Rep. Congo": "COD",
    "Congo, Democratic Republic of the": "COD",
    "DR Congo": "COD",
    "DRC": "COD",
    "Cote d'Ivoire": "CIV",
    "Côte d'Ivoire": "CIV",
    "Côte d’Ivoire": "CIV",
    "Ivory Coast": "CIV",
    "Djibouti": "DJI",
    "Egypt": "EGY",
    "Eswatini": "SWZ",
    "Swaziland": "SWZ",
    "Ethiopia": "ETH",
    "Gabon": "GAB",
    "Ghana": "GHA",
    "Guinea": "GIN",
    "Kenya": "KEN",
    "Lesotho": "LSO",
    "Liberia": "LBR",
    "Libya": "LBY",
    "Madagaskar": "MDG",
    "Madagascar": "MDG",
    "Malawi": "MWI",
    "Mauritania": "MRT",
    "Mauritius": "MUS",
    "Morocco": "MAR",
    "Mozambique": "MOZ",
    "Namibia": "NAM",
    "Nigeria": "NGA",
    "Rwanda": "RWA",
    "Sao Tome and Principe": "STP",
    "São Tomé and Príncipe": "STP",
    "Senegal": "SEN",
    "Seychelles": "SYC",
    "Sierra Leone": "SLE",
    "South Africa": "ZAF",
    "Tanzania": "TZA",
    "United Republic of Tanzania": "TZA",
    "The Gambia": "GMB",
    "Gambia": "GMB",
    "Togo": "TGO",
    "Tunisia": "TUN",
    "Uganda": "UGA",
    "Zambia": "ZMB",
    "Zimbabwe": "ZWE",

    # Latin America / Caribbean
    "Argentina": "ARG",
    "Barbados": "BRB",
    "Belize": "BLZ",
    "Bolivia": "BOL",
    "Brazil": "BRA",
    "Chile": "CHL",
    "Colombia": "COL",
    "Costa Rica": "CRI",
    "Cuba": "CUB",
    "Dominica": "DMA",
    "Dominican Republic": "DOM",
    "Dominican Rep.": "DOM",
    "Ecuador": "ECU",
    "El Salvador": "SLV",
    "Grenada": "GRD",
    "Guatemala": "GTM",
    "Guyana": "GUY",
    "Honduras": "HND",
    "Jamaica": "JAM",
    "Nicaragua": "NIC",
    "Panama": "PAN",
    "Paraguay": "PRY",
    "Peru": "PER",
    "Saint Lucia": "LCA",
    "Trinidad and Tobago": "TTO",
    "Uruguay": "URY",
}

def country_to_iso3(name):
    name = str(name).strip()

    if name in COUNTRY_TO_ISO3:
        return COUNTRY_TO_ISO3[name]

    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

# Compute modal import/export classes.
export_mode = df_no_self.apply(lambda row: mode_or_nan(row.values), axis=1)
import_mode = df_no_self.apply(lambda col: mode_or_nan(col.values), axis=0)

mode_df = pd.DataFrame({
    "country": df_no_self.index,
    "export_mode": export_mode.values,
    "import_mode": import_mode.reindex(df_no_self.index).values,
})

mode_df["iso_a3"] = mode_df["country"].map(country_to_iso3)

world["ne_iso_a3"] = world[iso_col].astype(str).str.strip()

# Remove Natural Earth pseudo-codes if present
world = world[world["ne_iso_a3"] != "-99"].copy()

merged = world.merge(
    mode_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

matrix_iso = set(mode_df["iso_a3"].dropna())
world_iso = set(world["ne_iso_a3"].dropna())

unmapped_iso = sorted(matrix_iso - world_iso)

print("\nMap matching diagnostics")
print("------------------------")
print(f"Countries in matrix: {mode_df.shape[0]}")
print(f"Countries with ISO-3 code: {mode_df['iso_a3'].notna().sum()}")
print(f"Countries matched to Natural Earth: {mode_df['iso_a3'].isin(world_iso).sum()}")
print(f"Countries without ISO-3 code: {mode_df['iso_a3'].isna().sum()}")

print("\nCountries without ISO-3 code:")
print(mode_df.loc[mode_df["iso_a3"].isna(), "country"].tolist())

print("\nCountries with ISO-3 code but not found in Natural Earth:")
print(mode_df.loc[mode_df["iso_a3"].isin(unmapped_iso), ["country", "iso_a3"]].to_string(index=False))

print("\nAfrican countries in matrix and their ISO codes:")
africa_check = [
    "Algeria", "Angola", "Benin", "Botswana", "Burkina Faso", "Cabo Verde",
    "Cameroon", "Comoros", "Republic of the Congo", "Cote d'Ivoire",
    "Democratic Republic of the Congo", "Dem. Rep. Congo", "Djibouti",
    "Egypt", "Eswatini", "Ethiopia", "Gabon", "Ghana", "Guinea", "Kenya",
    "Lesotho", "Liberia", "Libya", "Madagaskar", "Malawi", "Mauritania",
    "Mauritius", "Morocco", "Mozambique", "Namibia", "Nigeria", "Rwanda",
    "Sao Tome and Principe", "Senegal", "Seychelles", "Sierra Leone",
    "South Africa", "Tanzania", "The Gambia", "Togo", "Tunisia", "Uganda",
    "Zambia", "Zimbabwe"
]

print(
    mode_df[
        mode_df["country"].isin(africa_check)
    ][["country", "iso_a3", "import_mode", "export_mode"]].to_string(index=False)
)

### Interpretation and rationale: modal import and export maps

The mode maps translate the country-level profiles into geographic form. For the export map, each country is colored by the most frequent legal class in its outgoing row. For the import map, each country is colored by the most frequent legal class in its incoming column. Thus, the maps do not represent individual country pairs. Instead, they summarize many directed relationships into one dominant class per country.

These maps are useful for broad geographic orientation. They help reveal whether dominant regulatory profiles are spatially structured, for example whether groups of neighboring countries tend to share similar modal import or export classes. However, because the mode is a strong compression, two countries with the same modal class may still have very different distributions. A country that is 90% C and a country that is 40% C can both appear as C. The mode maps should therefore be interpreted alongside the mode-share, stacked-distribution, and entropy maps below.

Technically, the maps use ISO-3 country codes to join the matrix-derived summaries to Natural Earth country geometries. Countries present in the matrix but not matched to a geometry are reported by the diagnostic print statements.

In [ ]:
# Render import/export mode maps after ISO-based matching.

legend_handles = [
    mpatches.Patch(color=PALETTE[c], label=CATEGORY_LABELS[i])
    for i, c in enumerate(CATEGORY_ORDER)
]

def plot_mode_map(column, title, outfile_base):
    matched_geo = merged[merged[column].notna()].copy()

    print(f"\n{title}")
    print(f"Countries in matrix: {len(mode_df)}")
    print(f"Countries colored on map: {matched_geo['country'].nunique()}")

    missing = mode_df.loc[
        ~mode_df["iso_a3"].isin(set(matched_geo["ne_iso_a3"].dropna())),
        ["country", "iso_a3"]
    ]

    print("\nCountries from matrix not colored:")
    print(missing.to_string(index=False))

    if matched_geo.empty:
        print("No matched countries to plot.")
        return

    matched_geo["plot_color"] = matched_geo[column].map(PALETTE)

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        ax=ax,
        color=matched_geo["plot_color"],
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    xpad = (maxx - minx) * 0.05
    ypad = (maxy - miny) * 0.08

    ax.set_xlim(minx - xpad, maxx + xpad)
    ax.set_ylim(miny - ypad, maxy + ypad)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    ax.legend(
        handles=legend_handles,
        title="Dominant compatibility class",
        loc="lower left",
        frameon=True,
        fontsize=8,
        title_fontsize=9,
    )

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()


plot_mode_map(
    "import_mode",
    "Dominant import compatibility class by receiving jurisdiction",
    "03a_world_map_import_mode",
)

plot_mode_map(
    "export_mode",
    "Dominant export compatibility class by exporting jurisdiction",
    "03b_world_map_export_mode",
)

### Interpretation of the modal maps

If a country appears with the same color in the import and export maps, its dominant legal profile is similar as a sender and receiver. If the colors differ, that country may occupy an asymmetric governance position: for example, it may commonly export under one class while receiving data under another.

Gray countries indicate that the country is present in the geographic boundary file but not matched to the matrix, or that no modal class could be assigned. If many relevant countries are gray, inspect the printed unmatched-country list and extend the `NAME_TO_NATURAL_EARTH` dictionary. Map interpretation is only as good as the country-name harmonization.

### Interpretation and rationale: mode-share confidence maps

The modal maps identify each country’s dominant class, but they do not indicate how dominant that class is. The mode-share maps address this limitation by plotting the proportion of a country’s relationships that fall into its modal class. A value near 1 indicates that the modal class is a strong and representative summary of that country’s profile. A lower value indicates that the mode is weak and that the country’s relationships are spread across multiple legal classes.

These maps should be read as confidence or reliability maps for the modal summaries. If a country has a modal class of C but a low mode share, then the label C should be interpreted cautiously because the country may have substantial numbers of B, D, or other classes as well. Conversely, high mode share indicates a more homogeneous legal profile.

This section therefore qualifies the preceding mode maps. It asks not only “what is the dominant class?” but also “how much of the country’s profile does that dominant class actually explain?”

In [ ]:
# 1. Mode-share confidence maps
# These maps quantify how dominant the modal class is for each country.

def mode_and_share(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return pd.Series({"mode": np.nan, "mode_share": np.nan})
    counts = values.value_counts(normalize=True)
    return pd.Series({
        "mode": counts.index[0],
        "mode_share": counts.iloc[0],
    })

# Export statistics: one row per origin/exporting country.
export_mode_stats = df_no_self.apply(
    lambda row: mode_and_share(row.values),
    axis=1,
)

# Import statistics: one row per destination/receiving country.
# We build this explicitly from columns to avoid orientation errors.
import_mode_stats = pd.DataFrame(
    [
        mode_and_share(df_no_self[col].values)
        for col in df_no_self.columns
    ],
    index=df_no_self.columns,
)

mode_confidence_df = pd.DataFrame({
    "country": df_no_self.index,
    "export_mode": export_mode_stats.reindex(df_no_self.index)["mode"].values,
    "export_mode_share": export_mode_stats.reindex(df_no_self.index)["mode_share"].values,
    "import_mode": import_mode_stats.reindex(df_no_self.index)["mode"].values,
    "import_mode_share": import_mode_stats.reindex(df_no_self.index)["mode_share"].values,
})

mode_confidence_df["iso_a3"] = mode_confidence_df["country"].map(country_to_iso3)

mode_confidence_merged = world.merge(
    mode_confidence_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

def plot_mode_share_map(column, title, outfile_base):
    matched_geo = mode_confidence_merged[mode_confidence_merged[column].notna()].copy()

    print(f"\n{title}")
    print(f"Countries colored on map: {matched_geo['country'].nunique()}")

    if matched_geo.empty:
        print("No matched countries to plot.")
        return

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        column=column,
        ax=ax,
        cmap="viridis",
        legend=True,
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
        vmin=0,
        vmax=1,
        legend_kwds={"label": "Mode share"},
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    ax.set_xlim(minx - (maxx - minx) * 0.05, maxx + (maxx - minx) * 0.05)
    ax.set_ylim(miny - (maxy - miny) * 0.08, maxy + (maxy - miny) * 0.08)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()

plot_mode_share_map(
    "import_mode_share",
    "Import-mode dominance: proportion of incoming relationships in modal class",
    "03c_world_map_import_mode_share",
)

plot_mode_share_map(
    "export_mode_share",
    "Export-mode dominance: proportion of outgoing relationships in modal class",
    "03d_world_map_export_mode_share",
)


### Interpretation and rationale: mean legal-friction maps

The mean-friction maps provide a different type of country-level summary. Instead of using the most frequent class, they convert the ordered legal categories into an ordinal friction scale, with A1 as lowest friction and E as highest friction, and then compute the average incoming or outgoing friction for each country.

These maps are useful when the question concerns the overall severity of legal difficulty rather than the dominant category. A country may have a modal class of C, but its mean friction can distinguish whether its non-modal relationships are mostly easier, such as B, or harder, such as D and E. The mean-friction maps therefore capture information that the mode maps discard.

Because the legal classes are ordinal rather than truly continuous, the mean should be interpreted as a pragmatic summary index rather than a precise metric quantity. It is useful for comparing broad levels of relative friction, especially when paired with the full matrix and stacked profiles.

In [ ]:
# 2. Mean friction maps
# These maps summarize each country's average incoming or outgoing legal friction.

mean_friction_df = pd.DataFrame({
    "country": numeric_no_self.index,
    "export_mean_friction": numeric_no_self.mean(axis=1, skipna=True).values,
    "import_mean_friction": numeric_no_self.mean(axis=0, skipna=True).reindex(numeric_no_self.index).values,
})

mean_friction_df["iso_a3"] = mean_friction_df["country"].map(country_to_iso3)

mean_friction_merged = world.merge(
    mean_friction_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

def plot_mean_friction_map(column, title, outfile_base):
    matched_geo = mean_friction_merged[mean_friction_merged[column].notna()].copy()

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        column=column,
        ax=ax,
        cmap="magma_r",
        legend=True,
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
        vmin=0,
        vmax=5,
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    ax.set_xlim(minx - (maxx - minx) * 0.05, maxx + (maxx - minx) * 0.05)
    ax.set_ylim(miny - (maxy - miny) * 0.08, maxy + (maxy - miny) * 0.08)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()

plot_mean_friction_map(
    "import_mean_friction",
    "Mean incoming legal friction by receiving jurisdiction",
    "03e_world_map_import_mean_friction",
)

plot_mean_friction_map(
    "export_mean_friction",
    "Mean outgoing legal friction by exporting jurisdiction",
    "03f_world_map_export_mean_friction",
)

### Interpretation and rationale: entropy maps

The entropy maps quantify heterogeneity in each country’s legal profile. Entropy is high when a country’s relationships are spread across several compatibility classes and low when most relationships fall into one class. In this context, entropy does not indicate better or worse interoperability; it indicates diversity or inconsistency of legal classifications across partners.

These maps complement both the mode and mean-friction maps. A country can have moderate mean friction and still have either low entropy, if most relationships are consistently C, or high entropy, if it has a mixture of A, B, C, D, and E relationships. Thus, entropy helps distinguish uniform regulatory profiles from heterogeneous ones.

Interpretively, high entropy may indicate complex or partner-dependent governance conditions, whereas low entropy may indicate a more uniform legal regime. These maps are particularly useful for identifying countries whose single-color modal summary is potentially misleading.

In [ ]:
# 3. Entropy maps
# Entropy quantifies heterogeneity of a country's import/export compatibility profile.

def shannon_entropy(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan
    p = values.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

entropy_map_df = pd.DataFrame({
    "country": df_no_self.index,
    "export_entropy": df_no_self.apply(lambda row: shannon_entropy(row.values), axis=1).values,
    "import_entropy": df_no_self.apply(lambda col: shannon_entropy(col.values), axis=0).reindex(df_no_self.index).values,
})

entropy_map_df["iso_a3"] = entropy_map_df["country"].map(country_to_iso3)

entropy_merged = world.merge(
    entropy_map_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

def plot_entropy_map(column, title, outfile_base):
    matched_geo = entropy_merged[entropy_merged[column].notna()].copy()

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        column=column,
        ax=ax,
        cmap="cividis",
        legend=True,
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    ax.set_xlim(minx - (maxx - minx) * 0.05, maxx + (maxx - minx) * 0.05)
    ax.set_ylim(miny - (maxy - miny) * 0.08, maxy + (maxy - miny) * 0.08)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()

plot_entropy_map(
    "import_entropy",
    "Import-profile entropy by receiving jurisdiction",
    "03g_world_map_import_entropy",
)

plot_entropy_map(
    "export_entropy",
    "Export-profile entropy by exporting jurisdiction",
    "03h_world_map_export_entropy",
)

In [ ]:
# Entropy maps, website style (Figure 3d). Same numbers as above: Shannon entropy in bits over
# the six classes of each jurisdiction's complete incoming (or outgoing) profile, zero-frequency
# classes omitted, not normalised. Colour scale runs from 0 to the maximum observed.
from matplotlib.colors import LinearSegmentedColormap
SITE_VIRIDIS = LinearSegmentedColormap.from_list("site_viridis", ["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"])

def plot_entropy_map_website(column, caption, outfile_base):
    matched_geo = entropy_merged[entropy_merged[column].notna()].copy()
    emax = float(entropy_map_df[column].max())
    colors = [SITE_VIRIDIS(v / emax) for v in matched_geo[column]]

    def legend(ax):
        cax = ax.inset_axes([0.47, 0.08, 0.22, 0.025])
        sm = plt.cm.ScalarMappable(cmap=SITE_VIRIDIS, norm=plt.Normalize(0, emax))
        cb = plt.colorbar(sm, cax=cax, orientation="horizontal", ticks=[0, emax])
        cb.ax.set_xticklabels(["0 bits", f"{emax:.2f} bits"], fontsize=8); cb.outline.set_visible(False)
        cax.set_title("Entropy of " + ("incoming" if column.startswith("import") else "outgoing") + " conditions",
                      fontsize=8, loc="left", color=MUTED, pad=3)
        ax.legend(handles=class_legend_handles([], not_coded=True), loc="lower left", bbox_to_anchor=(0.43, 0.0),
                  frameon=False, fontsize=8)

    website_map(matched_geo, colors, caption, legend, outfile_base)

plot_entropy_map_website(
    "import_entropy",
    "Shannon entropy over the six classes of each jurisdiction's complete incoming profile; "
    "scale runs to the maximum observed (theoretical maximum log2 6 = 2.585 bits).",
    "03g_world_map_import_entropy_publication",
)
plot_entropy_map_website(
    "export_entropy",
    "Shannon entropy over the six classes of each jurisdiction's complete outgoing profile; "
    "scale runs to the maximum observed (theoretical maximum log2 6 = 2.585 bits).",
    "03h_world_map_export_entropy_publication",
)


### Interpretation and rationale: reference-region compatibility maps

The first cell below computes the reference-set modes and draws exploratory maps; the second draws the publication versions in the website style (Figure 3c).

The preceding maps summarize each country across all partners. The reference-region maps return to a more concrete relational question: how compatible is each country with a specified partner set? Here the reference set is Europe/UK, and the maps summarize each country’s dominant compatibility class either for exports to that reference set or for imports from that reference set.

These maps are closer to the original pairwise matrix than the global mode maps because they restrict the summary to a clearly defined set of partners. They are therefore easier to interpret in policy terms. For example, an export-to-Europe/UK map asks how each country is generally positioned when sending data to Europe/UK jurisdictions. The import-from-Europe/UK map asks the reciprocal question of how each country is positioned when receiving data from that reference set.

These maps are especially useful when the manuscript needs to discuss a specific governance bloc, such as Europe/UK, rather than global averages.

In [ ]:
# 4. Specific-reference maps
# These maps show compatibility with a chosen partner country or reference region.

REFERENCE_COUNTRIES = [
    "United Kingdom",
    "Germany",
    "France",
    "Italy",
    "Spain",
    "Netherlands",
    "Belgium",
    "Sweden",
    "Austria",
    "Finland",
    "Ireland",
    "Denmark",
    "Luxembourg",
    "Czechia",
    "Portugal",
    "Lithuania",
    "Estonia",
    "Poland",
    "Greece",
    "Hungary",
    "Slovakia",
    "Slovenia",
    "Romania",
    "Switzerland",
]

REFERENCE_COUNTRIES = [c for c in REFERENCE_COUNTRIES if c in df_no_self.index and c in df_no_self.columns]

print("Reference countries retained:")
print(REFERENCE_COUNTRIES)

def mode_across_values(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan
    return values.value_counts().idxmax()

reference_df = pd.DataFrame({"country": df_no_self.index})

# Country -> reference region:
# For each origin country, summarize outgoing compatibility to the reference set.
reference_df["export_to_reference_mode"] = [
    mode_across_values(df_no_self.loc[country, REFERENCE_COUNTRIES].values)
    if country in df_no_self.index else np.nan
    for country in df_no_self.index
]

# Reference region -> country:
# For each destination country, summarize incoming compatibility from the reference set.
reference_df["import_from_reference_mode"] = [
    mode_across_values(df_no_self.loc[REFERENCE_COUNTRIES, country].values)
    if country in df_no_self.columns else np.nan
    for country in df_no_self.index
]

reference_df["iso_a3"] = reference_df["country"].map(country_to_iso3)

reference_merged = world.merge(
    reference_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

def plot_reference_mode_map(column, title, outfile_base):
    matched_geo = reference_merged[reference_merged[column].notna()].copy()
    matched_geo["plot_color"] = matched_geo[column].map(PALETTE)

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        ax=ax,
        color=matched_geo["plot_color"],
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    ax.set_xlim(minx - (maxx - minx) * 0.05, maxx + (maxx - minx) * 0.05)
    ax.set_ylim(miny - (maxy - miny) * 0.08, maxy + (maxy - miny) * 0.08)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    legend_handles = [
        mpatches.Patch(color=PALETTE[c], label=CATEGORY_LABELS[i])
        for i, c in enumerate(CATEGORY_ORDER)
    ]

    ax.legend(
        handles=legend_handles,
        title="Compatibility class",
        loc="lower left",
        frameon=True,
        fontsize=8,
        title_fontsize=9,
    )

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()

plot_reference_mode_map(
    "export_to_reference_mode",
    "Dominant compatibility for exports to Europe/UK reference set",
    "03i_world_map_export_to_europeuk_reference",
)

plot_reference_mode_map(
    "import_from_reference_mode",
    "Dominant compatibility for imports from Europe/UK reference set",
    "03j_world_map_import_from_europeuk_reference",
)

In [ ]:
# Reference-set maps, website style (Figure 3c). For each destination, the modal class of its
# incoming pathways from the 24-jurisdiction European reference set (23 for members of the set);
# and, for each origin, the modal class of its outgoing pathways to the set.
# Uses reference_df / reference_merged computed in the previous cell and the website PALETTE.

def plot_reference_mode_map_website(column, caption, outfile_base):
    matched_geo = reference_merged[reference_merged[column].notna()].copy()
    colors = matched_geo[column].map(PALETTE).tolist()

    def legend(ax):
        ax.legend(handles=class_legend_handles(not_coded=True), loc="lower left", bbox_to_anchor=(0.43, 0.02),
                  frameon=False, fontsize=8)

    website_map(matched_geo, colors, caption, legend, outfile_base)

plot_reference_mode_map_website(
    "export_to_reference_mode",
    "Modal class of outgoing pathways to the 24-jurisdiction European reference set (23 for members of the set).",
    "03i_world_map_export_to_europeuk_reference_publication",
)
plot_reference_mode_map_website(
    "import_from_reference_mode",
    "Modal class of incoming pathways from the 24-jurisdiction European reference set (23 for members of the set).",
    "03j_world_map_import_from_europeuk_reference_publication",
)


## 3b. Network of enabled pathways (Figure 3b, website style)

The directional matrix is thresholded to the enabled classes A1, A2 and B, weighted 3, 2 and 1. Jurisdictions are placed on three regional rings (position on a ring carries no meaning), edges are coloured by class, and node size is the sum of weighted in-strength and out-strength, as in the manuscript Methods. This replaces the force-directed layout used in earlier drafts: the ring layout keeps the regional structure legible and matches the interactive version on braindatagovernance.org.

In [ ]:
# Network of enabled pathways (A1/A2/B), website style
from matplotlib.lines import Line2D

EDGE_WEIGHT = {"A1": 3, "A2": 2, "B": 1}
countries = list(df_no_self.index)

out_strength = {c: int(sum(EDGE_WEIGHT.get(v, 0) for v in df_no_self.loc[c].dropna())) for c in countries}
in_strength = {c: int(sum(EDGE_WEIGHT.get(v, 0) for v in df_no_self[c].dropna())) for c in countries}
network_df = pd.DataFrame({"country": countries, "region": [region_of(c) for c in countries],
                           "weighted_out_strength": [out_strength[c] for c in countries],
                           "weighted_in_strength": [in_strength[c] for c in countries]})
n_edges = {c: int((df_no_self == c).sum().sum()) for c in EDGE_WEIGHT}
print("Retained edges:", n_edges, "total", sum(n_edges.values()))
display(network_df.sort_values("weighted_out_strength", ascending=False).head(10))

# ring positions (same geometry as the website, in a 900 x 620 frame)
centers = {"Europe": (250, 300), "Africa": (640, 220), "Latin America & Caribbean": (620, 470)}
radius = {"Europe": 150, "Africa": 170, "Latin America & Caribbean": 120}
pos = {}
for r in REGIONS:
    members = [c for c in countries if region_of(c) == r]
    for k, c in enumerate(members):
        a = 2 * math.pi * k / len(members)
        pos[c] = (centers[r][0] + radius[r] * math.cos(a), centers[r][1] + radius[r] * math.sin(a))

fig, ax = plt.subplots(figsize=(12, 8.3), dpi=300)
for cls in ["B", "A2", "A1"]:          # B first so the enabled pathways sit on top
    for a in countries:
        for b in countries:
            if a != b and df_no_self.loc[a, b] == cls:
                ax.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]], color=PALETTE[cls],
                        lw=0.5 if cls == "B" else 1.2, alpha=0.15, solid_capstyle="round", zorder=1)
smax = max(out_strength[c] + in_strength[c] for c in countries)
for c in countries:
    s = out_strength[c] + in_strength[c]
    ax.add_patch(plt.Circle(pos[c], 3 + 9 * math.sqrt(s / smax), facecolor=REGION_COL[region_of(c)],
                            edgecolor="white", lw=1, zorder=3))
for r in REGIONS:   # label above the ring; below it for the Latin American ring, which overlaps the African one
    below = r == "Latin America & Caribbean"
    ax.text(centers[r][0], centers[r][1] + (radius[r] + 16) * (1 if below else -1), r, ha="center",
            va="top" if below else "bottom", fontsize=11, fontweight="600")
ax.set_xlim(0, 900); ax.set_ylim(620, 0); ax.set_aspect("equal"); ax.axis("off")
ax.legend(handles=[Line2D([], [], color=PALETTE[c], lw=2 if c != "B" else 1,
                          label=f"{CATEGORY_LABELS[CATEGORY_ORDER.index(c)]} (weight {EDGE_WEIGHT[c]})") for c in EDGE_WEIGHT]
                  + [Line2D([], [], marker="o", ls="", color=REGION_COL[r], markeredgecolor="white", markersize=9, label=r) for r in REGIONS],
          loc="lower left", frameon=False, fontsize=8, ncol=2)
ax.text(0, 640, "Pathways of class C, D and E are omitted. Node size is the sum of weighted in- and out-strength; "
                "position on the ring carries no meaning.", color=MUTED, fontsize=8, va="top")
savefig(fig, "03k_network_enabled_pathways")
plt.show()


## 3c. Region-to-region corridors

Share of each class for every origin-region to destination-region corridor, aggregated from the matrix. This is the summary shown under "Region to region" on the website; it is not a panel of Figure 3.

In [ ]:
# Region-to-region class shares (website "Region to region" summary)
corridor = {}
for a in REGIONS:
    for b in REGIONS:
        sub = df_no_self.loc[[c for c in countries if region_of(c) == a], [c for c in countries if region_of(c) == b]]
        corridor[f"{a} → {b}"] = sub.stack().value_counts().reindex(CATEGORY_ORDER).fillna(0).astype(int)
corridor_df = pd.DataFrame(corridor).T
display(corridor_df)

fig, ax = plt.subplots(figsize=(9, 4.2), dpi=300)
for i, (k, o) in enumerate(corridor_df.iterrows()):
    t = o.sum(); left = 0
    for c in CATEGORY_ORDER:
        if o[c]:
            ax.barh(i, 100 * o[c] / t, left=left, color=PALETTE[c], height=0.55); left += 100 * o[c] / t
    ax.text(101, i, " · ".join(f"{c} {100*o[c]/t:.1f}%" for c in CATEGORY_ORDER if o[c]), va="center", fontsize=7.5, color=MUTED)
ax.set_yticks(range(len(corridor_df))); ax.set_yticklabels(corridor_df.index, fontsize=8.5)
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xticks([]); ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.legend(handles=class_legend_handles(), loc="upper center", bbox_to_anchor=(0.35, -0.02), ncol=3, frameon=False, fontsize=8)
savefig(fig, "03l_region_corridors")
plt.show()

# matrix.json in the format read by the website (src/landscape/findings.py of pestillilab/braindatagovernance)
import json
# the website uses full jurisdiction names where this notebook uses Natural Earth short forms
SITE_NAME = {"Dem. Rep. Congo": "Democratic Republic of the Congo", "Congo": "Republic of the Congo",
             "eSwatini": "Eswatini", "Côte d'Ivoire": "Cote d'Ivoire", "Dominican Rep.": "Dominican Republic"}
S = lambda c: SITE_NAME.get(c, c)
eu = [c for c in countries if region_of(c) == "Europe"]
ent = entropy_map_df.set_index("country")["import_entropy"]
site = {
    "names": [S(c) for c in countries],
    "region": {S(c): region_of(c) for c in countries},
    "matrix": ["".join("." if pd.isna(v) else str(CATEGORY_ORDER.index(v)) for v in df_no_self.loc[c]) for c in countries],
    "importEU": {S(c): df_no_self.loc[[o for o in eu if o != c], c].value_counts().idxmax() for c in countries},
    "entropy": {S(c): round(float(ent[c]), 3) for c in countries},
    "corridor": {k.replace(" → ", "→"): {c: int(v) for c, v in o.items() if v} for k, o in corridor_df.iterrows()},
    "outStrength": {S(c): out_strength[c] for c in countries},
    "inStrength": {S(c): in_strength[c] for c in countries},
}
json.dump(site, open(OUTDIR / "matrix.json", "w"), ensure_ascii=False)
print("wrote", OUTDIR / "matrix.json")


## 4. Asymmetry matrix

The previous sections examined export and import profiles separately. This section directly compares the two directions for every country pair. The asymmetry matrix asks whether the transfer from country *i* to country *j* has the same legal classification as the transfer from country *j* to country *i*.

The analysis computes:

\[
\Delta_{ij} = f(i \rightarrow j) - f(j \rightarrow i)
\]

where \(f\) maps A1 to 0 and E to 5. A value of 0 indicates reciprocity. A positive value means the row-to-column transfer is more difficult than the reverse direction. A negative value means it is easier than the reverse direction.

This section is crucial because a directional matrix may look similar to a symmetric matrix unless asymmetry is explicitly quantified. The diagnostic counts printed before the plot are as important as the plot itself: they reveal how common asymmetry is and how large it is.

### Interpretation and rationale: directional asymmetry matrix

After characterizing the directional matrix itself, it is important to ask whether the legal classification from A to B is the same as from B to A. The asymmetry matrix computes the difference between each directed pair and its reverse direction. A value of zero means the relationship is reciprocal; a nonzero value means that one direction is more legally difficult than the other.

This analysis is essential because the matrix is directional. A symmetric-looking global pattern could still contain systematic one-way restrictions. The asymmetry matrix isolates those directional imbalances. In your diagnostics, most relationships were symmetric and all observed asymmetries were limited to one legal category. That result suggests a largely reciprocal structure with localized one-step directional differences.

This plot should be interpreted as a map of reciprocity and imbalance rather than as a direct compatibility matrix. Its primary question is: where does directionality matter?

In [ ]:
asymmetry = numeric - numeric.T
asymmetry = remove_self_transfers(asymmetry).apply(pd.to_numeric, errors="coerce")

# Diagnostic summary: check whether the matrix is truly symmetric or whether
# the visualization is hiding small directional differences.
asym_values = pd.Series(asymmetry.to_numpy(copy=True).flatten()).dropna()

print("Asymmetry value counts:")
print(asym_values.value_counts().sort_index())

print("\nAsymmetry summary:")
print(asym_values.describe())

print("\nProportion symmetric (Δ = 0):")
print((asym_values == 0).mean())

# Plot only integer asymmetry levels using a discrete diverging scale.
# This makes small ±1 or ±2 differences visible.
max_abs = int(np.nanmax(np.abs(asym_values)))
bounds = np.arange(-max_abs - 0.5, max_abs + 1.5, 1)

cmap = plt.get_cmap("RdBu_r", len(bounds) - 1)
norm_asym = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(20, 16), dpi=300)

im = ax.imshow(
    asymmetry.to_numpy(copy=True, dtype=float),
    cmap=cmap,
    norm=norm_asym,
    interpolation="none",
    aspect="auto",
)

ax.set_xticks(range(len(asymmetry.columns)))
ax.set_yticks(range(len(asymmetry.index)))
ax.set_xticklabels(asymmetry.columns, rotation=90, fontsize=7)
ax.set_yticklabels(asymmetry.index, fontsize=7)

cbar = fig.colorbar(
    im,
    ax=ax,
    ticks=np.arange(-max_abs, max_abs + 1, 1),
    fraction=0.046,
    pad=0.04,
)
cbar.set_label(
    "Directional asymmetry in legal friction\n"
    "Δ = friction(row → column) − friction(column → row)",
    fontsize=10,
)

ax.set_title(
    "Directional asymmetry matrix",
    fontsize=15,
    pad=18,
)
ax.set_xlabel("Destination jurisdiction", fontsize=11)
ax.set_ylabel("Origin jurisdiction", fontsize=11)

fig.tight_layout()
savefig(fig, "04_asymmetry_matrix_discrete")
plt.show()

### 4.2 Distribution of directional asymmetry values

The asymmetry matrix is the most precise way to inspect reciprocity at the pairwise level. The next step is to summarize how much asymmetry exists overall. This histogram counts the values of the asymmetry matrix after removing self-transfers. A value of zero indicates reciprocity; positive and negative values indicate that one direction is one or more legal categories more difficult than the reverse direction.

This plot is intentionally simple. It directly tests whether directional imbalance is a dominant feature of the matrix or whether most relationships are reciprocal. In this dataset, the earlier diagnostics showed that most entries are zero and that nonzero asymmetries are limited to one category. That pattern is much clearer in a histogram than in a country-level map.

In [ ]:
# Histogram of pairwise directional asymmetry values.

asymmetry = numeric - numeric.T
asymmetry = remove_self_transfers(asymmetry).apply(pd.to_numeric, errors="coerce")

asym_values = pd.Series(asymmetry.to_numpy(copy=True).flatten()).dropna()

print("Asymmetry value counts:")
print(asym_values.value_counts().sort_index())

print("\nAsymmetry summary:")
print(asym_values.describe())

print("\nProportion symmetric (Δ = 0):")
print((asym_values == 0).mean())

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

counts = asym_values.value_counts().sort_index()
ax.bar(
    counts.index.astype(str),
    counts.values,
    color="#7D8FA6",
    edgecolor="black",
    linewidth=0.6,
)

ax.set_title(
    "Distribution of directional asymmetry values",
    fontsize=14,
    pad=14,
)
ax.set_xlabel("Directional asymmetry value: friction(row → column) − friction(column → row)")
ax.set_ylabel("Number of directed country pairs")

for x, y in zip(range(len(counts)), counts.values):
    ax.text(x, y, f"{int(y)}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
savefig(fig, "04b_asymmetry_value_histogram")
plt.show()


### 4.3 Regional asymmetry matrix

Because country-level asymmetry maps are difficult to interpret and can be mathematically redundant, the more informative aggregation is regional. This regional asymmetry matrix summarizes directional imbalance between broad geopolitical regions. Each cell reports the mean signed asymmetry from an origin region to a destination region. Positive values mean that transfers from the row region to the column region are, on average, more difficult than the reverse direction. Negative values mean the opposite.

This plot answers the policy-relevant question more directly than a country-level asymmetry map: where are directional imbalances concentrated? It also retains directionality while reducing the complexity of the full country-by-country matrix. The regional grouping should be reviewed and adjusted to match the manuscript’s conceptual framework, but the structure of the analysis is general.

In [ ]:
# Regional asymmetry matrix

# Define broad regions. Edit this mapping if the manuscript uses different regional definitions.
EUROPE_UK = {
    "Germany", "France", "Italy", "Spain", "Netherlands", "Belgium", "Sweden", "Austria",
    "Finland", "Ireland", "Denmark", "Luxembourg", "Czechia", "Portugal", "Lithuania",
    "Estonia", "Poland", "Greece", "Hungary", "Slovakia", "Slovenia", "Romania",
    "United Kingdom", "Switzerland", "Norway", "Iceland", "Croatia", "Bulgaria",
    "Latvia", "Malta", "Cyprus"
}

NORTH_AMERICA = {
    "United States", "United States of America", "Canada", "Mexico"
}

LATIN_AMERICA = {
    "Argentina", "Barbados", "Belize", "Bolivia", "Brazil", "Chile", "Colombia",
    "Costa Rica", "Cuba", "Dominica", "Dominican Republic", "Dominican Rep.",
    "Ecuador", "El Salvador", "Grenada", "Guatemala", "Guyana", "Honduras",
    "Jamaica", "Nicaragua", "Panama", "Paraguay", "Peru", "Saint Lucia",
    "Trinidad and Tobago", "Uruguay"
}

AFRICA = set(df_no_self.index) - EUROPE_UK - NORTH_AMERICA - LATIN_AMERICA

def infer_region_for_asymmetry(country):
    if country in EUROPE_UK:
        return "Europe/UK"
    if country in NORTH_AMERICA:
        return "North America"
    if country in LATIN_AMERICA:
        return "Latin America"
    if country in AFRICA:
        return "Africa"
    return "Other"

region_map = {country: infer_region_for_asymmetry(country) for country in asymmetry.index}
regions = ["Africa", "Europe/UK", "Latin America", "North America"]
regions = [r for r in regions if any(region_map[c] == r for c in asymmetry.index)]

region_asymmetry = pd.DataFrame(index=regions, columns=regions, dtype=float)
region_asymmetry_counts = pd.DataFrame(index=regions, columns=regions, dtype=float)

for origin_region in regions:
    origin_countries = [c for c in asymmetry.index if region_map[c] == origin_region]
    for dest_region in regions:
        dest_countries = [c for c in asymmetry.columns if region_map[c] == dest_region]
        values = asymmetry.loc[origin_countries, dest_countries].to_numpy(copy=True).flatten()
        values = pd.Series(values).dropna().astype(float)
        region_asymmetry.loc[origin_region, dest_region] = values.mean() if len(values) else np.nan
        region_asymmetry_counts.loc[origin_region, dest_region] = len(values)

display(region_asymmetry)

vmax = np.nanmax(np.abs(region_asymmetry.to_numpy(dtype=float)))

fig, ax = plt.subplots(figsize=(8, 7), dpi=300)

im = ax.imshow(
    region_asymmetry.to_numpy(dtype=float),
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
    interpolation="none",
)

ax.set_xticks(range(len(regions)))
ax.set_yticks(range(len(regions)))
ax.set_xticklabels(regions, rotation=45, ha="right")
ax.set_yticklabels(regions)

for i, origin_region in enumerate(regions):
    for j, dest_region in enumerate(regions):
        val = region_asymmetry.iloc[i, j]
        n = region_asymmetry_counts.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.3f}\n(n={int(n)})", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(
    "Mean signed asymmetry\npositive = row region harder as exporter",
    fontsize=10,
)

ax.set_title(
    "Regional directional asymmetry matrix",
    fontsize=14,
    pad=14,
)
ax.set_xlabel("Destination region")
ax.set_ylabel("Origin region")

fig.tight_layout()
savefig(fig, "04c_regional_asymmetry_matrix")
plt.show()


### 4.4 Optional sparse asymmetry network

The dense country-level network of all easy-transfer edges was not interpretable because the graph was too connected. A more meaningful network representation is to draw only asymmetric relationships. This graph is much sparser and focuses on the specific directional imbalances identified by the asymmetry matrix.

In this network, nodes are countries and directed edges point from the country for which the transfer direction is harder toward the counterpart. Because asymmetry is rare and bounded in this dataset, the network should be treated as an exploratory diagnostic rather than a primary figure. It is most useful for identifying whether a few countries or regions account for many of the asymmetric relationships.

This updated version labels every country node, colors countries by broad region, scales node size by betweenness/asymmetry degree, and exports both raster and vector versions for manuscript workflows.

Rendering note: the code now calls both `plt.show()` and an explicit `IPython.display.Image` fallback after saving the PNG, so the network should remain visible across JupyterLab, VS Code, nbviewer, and exported HTML.

In [ ]:
# Optional sparse asymmetry network with full country labels

import networkx as nx
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from IPython.display import display, Image as IPythonImage

G_asym = nx.DiGraph()

for country in asymmetry.index:
    G_asym.add_node(country)

for origin in asymmetry.index:
    for dest in asymmetry.columns:
        val = asymmetry.loc[origin, dest]
        if pd.notna(val) and val > 0:
            # origin -> dest is harder than dest -> origin.
            G_asym.add_edge(origin, dest, weight=float(val))

asym_degree_df = pd.DataFrame({
    "country": list(G_asym.nodes),
    "out_asymmetry_count": [G_asym.out_degree(n) for n in G_asym.nodes],
    "in_asymmetry_count": [G_asym.in_degree(n) for n in G_asym.nodes],
    "total_asymmetry_degree": [G_asym.out_degree(n) + G_asym.in_degree(n) for n in G_asym.nodes],
}).sort_values("out_asymmetry_count", ascending=False)

display(asym_degree_df.head(20))

# Node-level centrality summaries for visualization and later interpretation.
# Betweenness emphasizes countries that connect otherwise separate parts of the asymmetry graph.
if G_asym.number_of_edges() > 0:
    betweenness = nx.betweenness_centrality(G_asym, weight="weight", normalized=True)
else:
    betweenness = {n: 0 for n in G_asym.nodes}

nx.set_node_attributes(G_asym, betweenness, "betweenness")

node_sizes = [
    120 + 2500 * betweenness.get(n, 0) + 25 * (G_asym.out_degree(n) + G_asym.in_degree(n))
    for n in G_asym.nodes
]

# Reuse the region definitions from Section 4.3 when available.
# If the upstream region function is not defined, all nodes fall back to one neutral category.
try:
    node_regions = {n: infer_region_for_asymmetry(n) for n in G_asym.nodes}
except NameError:
    node_regions = {n: "Country" for n in G_asym.nodes}

region_palette = {
    "Africa / other": "#4C78A8",
    "Africa": "#4C78A8",
    "Europe/UK": "#F58518",
    "North America": "#54A24B",
    "Latin America": "#E45756",
    "Country": "#7D8FA6",
}
node_colors = [region_palette.get(node_regions[n], "#7D8FA6") for n in G_asym.nodes]

# Kamada-Kawai generally provides more separation than force-directed spring layouts
# for small-to-moderate country networks with labels on every node.
if G_asym.number_of_edges() > 0:
    pos = nx.kamada_kawai_layout(G_asym, weight="weight")
else:
    pos = nx.spring_layout(G_asym, k=1.8, seed=7)

edge_widths = [0.4 + 1.2 * G_asym[u][v].get("weight", 1.0) for u, v in G_asym.edges]

fig, ax = plt.subplots(figsize=(20, 18), dpi=300)

nx.draw_networkx_edges(
    G_asym,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=8,
    width=edge_widths,
    alpha=0.20,
    edge_color="0.35",
    connectionstyle="arc3,rad=0.06",
)

nx.draw_networkx_nodes(
    G_asym,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="white",
    linewidths=0.6,
    alpha=0.95,
)

# Label every country. A white stroke improves legibility when labels overlap edges.
for node, (x, y) in pos.items():
    ax.text(
        x,
        y,
        node,
        fontsize=7.5,
        ha="center",
        va="center",
        zorder=10,
        path_effects=[pe.withStroke(linewidth=2.6, foreground="white")],
    )

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        label=region,
        markerfacecolor=color,
        markeredgecolor="white",
        markersize=9,
    )
    for region, color in region_palette.items()
    if region in set(node_regions.values())
]

if legend_handles:
    ax.legend(
        handles=legend_handles,
        title="Region",
        loc="lower left",
        frameon=False,
        fontsize=9,
        title_fontsize=10,
    )

ax.set_title(
    "Sparse network of asymmetric transfer relationships",
    fontsize=16,
    pad=18,
)
ax.text(
    0.01,
    0.01,
    "Nodes are countries; directed edges point from the country for which transfer is harder toward the counterpart.\n"
    "All countries are labeled. Node color indicates region; node size reflects betweenness and asymmetry degree.",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
)
ax.axis("off")

fig.tight_layout()

# Save both a PNG and a vector PDF for manuscript workflows.
# The explicit IPython display call below is intentional: in some notebook viewers,
# large matplotlib network figures may save correctly but not render inline after
# cell execution. Displaying the saved PNG makes the plot visible reliably.
fig_name = "04d_sparse_asymmetry_network_labeled"
savefig(fig, fig_name)

# Keep a notebook-inline matplotlib rendering.
plt.show()

# Also display the saved PNG explicitly so the figure remains visible in JupyterLab,
# VS Code notebooks, nbviewer, and exported HTML.
png_path = OUTDIR / f"{fig_name}.png"
if png_path.exists():
    display(IPythonImage(filename=str(png_path)))
else:
    display(fig)


In [ ]:
# Publication-ready sparse asymmetry network with full country labels

import networkx as nx
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from IPython.display import display, Image as IPythonImage
import pandas as pd
import matplotlib.pyplot as plt

G_asym = nx.DiGraph()

# Add countries as nodes
for country in asymmetry.index:
    G_asym.add_node(country)

# Add asymmetric edges
for origin in asymmetry.index:
    for dest in asymmetry.columns:
        val = asymmetry.loc[origin, dest]
        if pd.notna(val) and val > 0:
            # origin -> dest is harder than dest -> origin
            G_asym.add_edge(origin, dest, weight=float(val))

# Summary table
asym_degree_df = pd.DataFrame({
    "country": list(G_asym.nodes),
    "out_asymmetry_count": [G_asym.out_degree(n) for n in G_asym.nodes],
    "in_asymmetry_count": [G_asym.in_degree(n) for n in G_asym.nodes],
    "total_asymmetry_degree": [
        G_asym.out_degree(n) + G_asym.in_degree(n)
        for n in G_asym.nodes
    ],
}).sort_values("out_asymmetry_count", ascending=False)

display(asym_degree_df.head(20))

# Betweenness centrality
if G_asym.number_of_edges() > 0:
    betweenness = nx.betweenness_centrality(
        G_asym,
        weight="weight",
        normalized=True
    )
else:
    betweenness = {n: 0 for n in G_asym.nodes}

nx.set_node_attributes(G_asym, betweenness, "betweenness")

# Degree
degree_dict = {
    n: G_asym.out_degree(n) + G_asym.in_degree(n)
    for n in G_asym.nodes
}

# More restrained node sizes
node_sizes = [
    150
    + 1500 * betweenness.get(n, 0)
    + 15 * degree_dict[n]
    for n in G_asym.nodes
]

# Region colors
try:
    node_regions = {n: infer_region_for_asymmetry(n) for n in G_asym.nodes}
except NameError:
    node_regions = {n: "Country" for n in G_asym.nodes}

region_palette = {
    "Africa / other": "#9C755F",
    "Africa": "#9C755F",
    "Europe/UK": "#4E79A7",
    "North America": "#59A14F",
    "Latin America": "#E15759",
    "Country": "#7D8FA6",
}

node_colors = [
    region_palette.get(node_regions[n], "#7D8FA6")
    for n in G_asym.nodes
]

# More spread-out layout
pos = nx.spring_layout(
    G_asym,
    k=4.5,
    iterations=500,
    seed=7,
    weight=None
)

# Lighter, more publication-friendly edges
edge_widths = [
    0.35 + 0.45 * G_asym[u][v].get("weight", 1.0)
    for u, v in G_asym.edges
]

fig, ax = plt.subplots(figsize=(20, 18), dpi=300)

# Edges
nx.draw_networkx_edges(
    G_asym,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=6,
    width=edge_widths,
    alpha=0.08,
    edge_color="0.45",
    connectionstyle="arc3,rad=0.06",
)

# Nodes
nx.draw_networkx_nodes(
    G_asym,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="white",
    linewidths=1.1,
    alpha=0.96,
)

# Labels: larger for more connected countries
for node, (x, y) in pos.items():
    deg = degree_dict[node]

    if deg >= 15:
        fs = 8.5
        fw = "bold"
    elif deg >= 5:
        fs = 7.5
        fw = "normal"
    else:
        fs = 6.3
        fw = "normal"

    ax.text(
        x,
        y,
        node,
        fontsize=fs,
        fontweight=fw,
        ha="center",
        va="center",
        zorder=10,
        path_effects=[
            pe.withStroke(linewidth=2.8, foreground="white")
        ],
    )

# Region legend
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        label=region,
        markerfacecolor=color,
        markeredgecolor="white",
        markersize=10,
    )
    for region, color in region_palette.items()
    if region in set(node_regions.values())
]

if legend_handles:
    legend = ax.legend(
        handles=legend_handles,
        title="Region",
        loc="lower left",
        frameon=False,
        fontsize=10,
        title_fontsize=11,
    )
    legend.get_title().set_fontweight("bold")

ax.set_title(
    "Cross-Border Transfer Network",
    fontsize=18,
    fontweight="bold",
    pad=20,
)

ax.axis("off")

fig.tight_layout()

fig_name = "04d_sparse_asymmetry_network_publication_ready"
savefig(fig, fig_name)

plt.show()

png_path = OUTDIR / f"{fig_name}.png"
if png_path.exists():
    display(IPythonImage(filename=str(png_path)))
else:
    display(fig)

### Interpretation of the asymmetry analysis

The first interpretation should come from the value counts. If most values are 0, the global matrix is largely reciprocal. If nonzero values are common, the legal landscape contains substantial direction-specific imbalance. If the maximum asymmetry is only ±1, then directional differences exist but are modest in magnitude. If larger values appear, some country pairs differ sharply depending on transfer direction.

In the current analysis, the diagnostic output previously indicated that approximately 85% of directed pairs were symmetric and that all observed asymmetries were limited to ±1. Under that result, the appropriate interpretation is that directionality matters, but the magnitude of legal imbalance is bounded. The scientific question then becomes less about how large asymmetries are and more about **where** they occur: which jurisdictions, regions, or legal mechanisms account for the nonreciprocal relationships.

In [ ]:
# Sparse asymmetry network with country colors matching the clustered matrix figure
# No region colors and no country labels on nodes

import networkx as nx
from matplotlib.patches import Patch
from IPython.display import display, Image as IPythonImage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# ----------------------------
# Build asymmetry graph
# ----------------------------
G_asym = nx.DiGraph()

for country in asymmetry.index:
    G_asym.add_node(country)

for origin in asymmetry.index:
    for dest in asymmetry.columns:
        val = asymmetry.loc[origin, dest]
        if pd.notna(val) and val > 0:
            G_asym.add_edge(origin, dest, weight=float(val))

# ----------------------------
# Summary table
# ----------------------------
asym_degree_df = pd.DataFrame({
    "country": list(G_asym.nodes),
    "out_asymmetry_count": [G_asym.out_degree(n) for n in G_asym.nodes],
    "in_asymmetry_count": [G_asym.in_degree(n) for n in G_asym.nodes],
    "total_asymmetry_degree": [
        G_asym.out_degree(n) + G_asym.in_degree(n)
        for n in G_asym.nodes
    ],
}).sort_values("out_asymmetry_count", ascending=False)

display(asym_degree_df.head(20))

# ----------------------------
# Use same clustered country order as matrix figure
# ----------------------------
X = numeric_no_self.copy().astype(float)
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = list(X.index[row_order])
n = len(ordered_countries)

# Same softened turbo country colors as matrix figure
base = plt.cm.turbo(np.linspace(0, 1, n))
blend = 0.45

country_colors = base.copy()
country_colors[:, :3] = (
    (1 - blend) * base[:, :3] +
    blend * 1.0
)

country_color_map = {
    country: country_colors[i]
    for i, country in enumerate(ordered_countries)
}

node_colors = [
    country_color_map.get(node, "#7D8FA6")
    for node in G_asym.nodes
]

# ----------------------------
# Centrality and node sizing
# ----------------------------
if G_asym.number_of_edges() > 0:
    betweenness = nx.betweenness_centrality(
        G_asym,
        weight="weight",
        normalized=True
    )
else:
    betweenness = {node: 0 for node in G_asym.nodes}

degree_dict = {
    node: G_asym.out_degree(node) + G_asym.in_degree(node)
    for node in G_asym.nodes
}

node_sizes = [
    150
    + 1500 * betweenness.get(node, 0)
    + 15 * degree_dict[node]
    for node in G_asym.nodes
]

# ----------------------------
# Layout
# ----------------------------
pos = nx.spring_layout(
    G_asym,
    k=4.5,
    iterations=500,
    seed=7,
    weight=None
)

edge_widths = [
    0.35 + 0.45 * G_asym[u][v].get("weight", 1.0)
    for u, v in G_asym.edges
]

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(20, 18), dpi=300)

nx.draw_networkx_edges(
    G_asym,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=6,
    width=edge_widths,
    alpha=0.08,
    edge_color="0.45",
    connectionstyle="arc3,rad=0.06",
)

nx.draw_networkx_nodes(
    G_asym,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="white",
    linewidths=1.1,
    alpha=0.96,
)

# ----------------------------
# Country legend using same order/colors as matrix
# ----------------------------
legend_elements = [
    Patch(
        facecolor=country_color_map[country],
        label=country
    )
    for country in ordered_countries
    if country in G_asym.nodes
]

legend = ax.legend(
    handles=legend_elements,
    title="Countries",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.04),
    frameon=False,
    ncol=6,
    columnspacing=1.1,
    handlelength=1.2,
    handletextpad=0.4,
    prop={"size": 18},
    title_fontproperties={"size": 20, "weight": "bold"},
)

ax.set_title(
    "Cross-Border Transfer Network",
    fontsize=20,
    fontweight="bold",
    pad=20,
)

ax.axis("off")
fig.tight_layout()

fig_name = "04d_sparse_asymmetry_network_country_colors_matching_matrix"
savefig(fig, fig_name)

plt.show()

png_path = OUTDIR / f"{fig_name}.png"
if png_path.exists():
    display(IPythonImage(filename=str(png_path)))
else:
    display(fig)

In [ ]:
# Sparse asymmetry network with country colors matching the clustered matrix figure
# Compact spring layout with reduced whitespace
# No region colors and no country labels on nodes

import networkx as nx
from matplotlib.patches import Patch
from IPython.display import display, Image as IPythonImage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# ----------------------------
# Build asymmetry graph
# ----------------------------
G_asym = nx.DiGraph()

for country in asymmetry.index:
    G_asym.add_node(country)

for origin in asymmetry.index:
    for dest in asymmetry.columns:
        val = asymmetry.loc[origin, dest]
        if pd.notna(val) and val > 0:
            G_asym.add_edge(origin, dest, weight=float(val))

# ----------------------------
# Summary table
# ----------------------------
asym_degree_df = pd.DataFrame({
    "country": list(G_asym.nodes),
    "out_asymmetry_count": [G_asym.out_degree(n) for n in G_asym.nodes],
    "in_asymmetry_count": [G_asym.in_degree(n) for n in G_asym.nodes],
    "total_asymmetry_degree": [
        G_asym.out_degree(n) + G_asym.in_degree(n)
        for n in G_asym.nodes
    ],
}).sort_values("out_asymmetry_count", ascending=False)

display(asym_degree_df.head(20))

# ----------------------------
# Use same clustered country order as matrix figure
# ----------------------------
X = numeric_no_self.copy().astype(float)
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = list(X.index[row_order])
n = len(ordered_countries)

# Same softened turbo country colors as matrix figure
base = plt.cm.turbo(np.linspace(0, 1, n))
blend = 0.45

country_colors = base.copy()
country_colors[:, :3] = (
    (1 - blend) * base[:, :3] +
    blend * 1.0
)

country_color_map = {
    country: country_colors[i]
    for i, country in enumerate(ordered_countries)
}

node_colors = [
    country_color_map.get(node, "#7D8FA6")
    for node in G_asym.nodes
]

# ----------------------------
# Centrality and node sizing
# ----------------------------
if G_asym.number_of_edges() > 0:
    betweenness = nx.betweenness_centrality(
        G_asym,
        weight="weight",
        normalized=True
    )
else:
    betweenness = {node: 0 for node in G_asym.nodes}

degree_dict = {
    node: G_asym.out_degree(node) + G_asym.in_degree(node)
    for node in G_asym.nodes
}

node_sizes = [
    150
    + 1500 * betweenness.get(node, 0)
    + 15 * degree_dict[node]
    for node in G_asym.nodes
]

# ----------------------------
# Compact layout
# ----------------------------
pos = nx.spring_layout(
    G_asym,
    k=2.2,              # lower = more compact; increase if too crowded
    iterations=800,
    seed=7,
    weight=None,
    scale=1.0
)

# Normalize coordinates so graph fills the plotting area
xy = np.array(list(pos.values()))
x = xy[:, 0]
y = xy[:, 1]

x_range = x.max() - x.min()
y_range = y.max() - y.min()

for node in pos:
    pos[node] = np.array([
        (pos[node][0] - x.mean()) / x_range,
        (pos[node][1] - y.mean()) / y_range
    ])

# Optional vertical compression
for node in pos:
    pos[node][1] *= 0.85

edge_widths = [
    0.35 + 0.45 * G_asym[u][v].get("weight", 1.0)
    for u, v in G_asym.edges
]

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(16, 16), dpi=300)

nx.draw_networkx_edges(
    G_asym,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=6,
    width=edge_widths,
    alpha=0.08,
    edge_color="0.45",
    connectionstyle="arc3,rad=0.06",
)

nx.draw_networkx_nodes(
    G_asym,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="white",
    linewidths=1.1,
    alpha=0.96,
)

# ----------------------------
# Country legend using same order/colors as matrix
# ----------------------------
legend_elements = [
    Patch(
        facecolor=country_color_map[country],
        label=country
    )
    for country in ordered_countries
    if country in G_asym.nodes
]

legend = ax.legend(
    handles=legend_elements,
    title="Countries",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.015),
    frameon=False,
    ncol=6,
    columnspacing=1.1,
    handlelength=1.2,
    handletextpad=0.4,
    prop={"size": 18},
    title_fontproperties={"size": 20, "weight": "bold"},
)

ax.set_title(
    "Cross-Border Transfer Network",
    fontsize=20,
    fontweight="bold",
    pad=20,
)

# Tight crop around graph
xy = np.array(list(pos.values()))
pad = 0.08
ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad)
ax.set_ylim(xy[:, 1].min() - pad, xy[:, 1].max() + pad)

ax.set_aspect("equal")
ax.margins(0)
ax.axis("off")

fig.tight_layout()

fig_name = "04d_sparse_asymmetry_network_country_colors_matching_matrix_compact"
savefig(fig, fig_name)

plt.show()

png_path = OUTDIR / f"{fig_name}.png"
if png_path.exists():
    display(IPythonImage(filename=str(png_path)))
else:
    display(fig)

In [ ]:
# Sparse asymmetry network with country colors matching the clustered matrix figure
# Keeps spring graph shape, reduces whitespace, and shortens visible edge lines
# No region colors and no country labels on nodes

import networkx as nx
from matplotlib.patches import Patch
from IPython.display import display, Image as IPythonImage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# ----------------------------
# Build asymmetry graph
# ----------------------------
G_asym = nx.DiGraph()

for country in asymmetry.index:
    G_asym.add_node(country)

for origin in asymmetry.index:
    for dest in asymmetry.columns:
        val = asymmetry.loc[origin, dest]
        if pd.notna(val) and val > 0:
            G_asym.add_edge(origin, dest, weight=float(val))

# ----------------------------
# Summary table
# ----------------------------
asym_degree_df = pd.DataFrame({
    "country": list(G_asym.nodes),
    "out_asymmetry_count": [G_asym.out_degree(n) for n in G_asym.nodes],
    "in_asymmetry_count": [G_asym.in_degree(n) for n in G_asym.nodes],
    "total_asymmetry_degree": [
        G_asym.out_degree(n) + G_asym.in_degree(n)
        for n in G_asym.nodes
    ],
}).sort_values("out_asymmetry_count", ascending=False)

display(asym_degree_df.head(20))

# ----------------------------
# Use same clustered country order as matrix figure
# ----------------------------
X = numeric_no_self.copy().astype(float)
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = list(X.index[row_order])
n = len(ordered_countries)

# Same softened turbo country colors as matrix figure
base = plt.cm.turbo(np.linspace(0, 1, n))
blend = 0.45

country_colors = base.copy()
country_colors[:, :3] = (
    (1 - blend) * base[:, :3] +
    blend * 1.0
)

country_color_map = {
    country: country_colors[i]
    for i, country in enumerate(ordered_countries)
}

node_colors = [
    country_color_map.get(node, "#7D8FA6")
    for node in G_asym.nodes
]

# ----------------------------
# Centrality and node sizing
# ----------------------------
if G_asym.number_of_edges() > 0:
    betweenness = nx.betweenness_centrality(
        G_asym,
        weight="weight",
        normalized=True
    )
else:
    betweenness = {node: 0 for node in G_asym.nodes}

degree_dict = {
    node: G_asym.out_degree(node) + G_asym.in_degree(node)
    for node in G_asym.nodes
}

node_sizes = [
    150
    + 1500 * betweenness.get(node, 0)
    + 15 * degree_dict[node]
    for node in G_asym.nodes
]

# ----------------------------
# Layout
# ----------------------------
# Keep original spread so center circles do NOT overlap more
pos = nx.spring_layout(
    G_asym,
    k=4.5,
    iterations=500,
    seed=7,
    weight=None
)

edge_widths = [
    0.35 + 0.45 * G_asym[u][v].get("weight", 1.0)
    for u, v in G_asym.edges
]

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(18, 18), dpi=300)

nx.draw_networkx_edges(
    G_asym,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=6,
    width=edge_widths,
    alpha=0.08,
    edge_color="0.45",
    connectionstyle="arc3,rad=0.06",
    min_source_margin=16,   # shortens visible lines without moving nodes
    min_target_margin=16,   # increase to 20–25 if you want even shorter lines
)

nx.draw_networkx_nodes(
    G_asym,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="white",
    linewidths=1.1,
    alpha=0.96,
)

# ----------------------------
# Country legend using same order/colors as matrix
# ----------------------------
legend_elements = [
    Patch(
        facecolor=country_color_map[country],
        label=country
    )
    for country in ordered_countries
    if country in G_asym.nodes
]

legend = ax.legend(
    handles=legend_elements,
    title="Countries",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.025),
    frameon=False,
    ncol=6,
    columnspacing=1.1,
    handlelength=1.2,
    handletextpad=0.4,
    prop={"size": 18},
    title_fontproperties={"size": 20, "weight": "bold"},
)

ax.set_title(
    "Cross-Border Transfer Network",
    fontsize=20,
    fontweight="bold",
    pad=20,
)

# ----------------------------
# Tight crop around graph without changing layout
# ----------------------------
xy = np.array(list(pos.values()))

pad_x = 0.04 * (xy[:, 0].max() - xy[:, 0].min())
pad_y = 0.04 * (xy[:, 1].max() - xy[:, 1].min())

ax.set_xlim(xy[:, 0].min() - pad_x, xy[:, 0].max() + pad_x)
ax.set_ylim(xy[:, 1].min() - pad_y, xy[:, 1].max() + pad_y)

ax.set_aspect("equal")
ax.margins(0)
ax.axis("off")

fig.tight_layout()

fig_name = "04d_sparse_asymmetry_network_country_colors_matching_matrix_short_edges"
savefig(fig, fig_name)

plt.show()

png_path = OUTDIR / f"{fig_name}.png"
if png_path.exists():
    display(IPythonImage(filename=str(png_path)))
else:
    display(fig)

In [ ]:
# Install networkx locally if needed:
# pip install networkx

try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False
    print("networkx is not installed. Network section skipped.")


### Interpretation of the directed network

Node position is determined by the network layout and should be interpreted cautiously. More important are the computed centrality measures. Out-strength summarizes how broadly a jurisdiction can export through feasible relationships. In-strength summarizes how broadly it can receive data through feasible relationships. Betweenness centrality identifies jurisdictions that lie on many shortest paths and may therefore function as bridges or intermediaries in the interoperability structure.

A dense cluster of countries with many A1/A2/B edges suggests a legal interoperability bloc. Peripheral countries with few feasible edges may represent jurisdictions where data-sharing requires additional legal infrastructure, policy harmonization, or case-specific agreements. The network is best treated as an exploratory tool for identifying hubs, bridges, and isolated jurisdictions, not as a substitute for the full matrix.

## 6. Country legal-profile embeddings and clustering

The previous section represented the data as a network of feasible directed transfers. This section returns to the country profiles themselves. Each country is treated as a vector of legal-friction values across all destinations. Countries with similar vectors have similar export profiles, even if they are geographically distant.

The goal is to discover whether the legal-interoperability matrix contains lower-dimensional structure: clusters, gradients, or latent axes of governance similarity. Several methods are included because each answers a slightly different question. PCA is linear and interpretable. Hierarchical clustering emphasizes group structure. MDS preserves pairwise distances. t-SNE and UMAP are nonlinear exploratory methods that may reveal local neighborhoods but require more caution in interpretation.

In [ ]:
# Install scikit-learn locally if needed:
# pip install scikit-learn

try:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("scikit-learn is not installed. PCA embedding skipped.")


### 6.1 PCA: linear embedding of outgoing legal profiles

PCA identifies orthogonal axes that explain the largest amount of variance in the country-by-destination profile matrix. If countries align along a single diagonal or line, that often means the dominant structure is close to one-dimensional: countries primarily differ by overall legal friction rather than by multiple independent patterns.

PCA is useful because the axes are mathematically transparent. However, when many country labels overlap, the plot may be difficult to read visually. In that case, the result is still informative: it suggests that PCA may be summarizing a broad gradient rather than revealing discrete clusters.

### Interpretation and rationale: PCA embedding of country profiles

The PCA embedding asks whether country profiles can be summarized along a small number of major axes. Each country is represented as a vector of outgoing legal-friction values, and PCA identifies orthogonal directions that explain the largest amount of variance across those profiles.

This method is useful as an initial diagnostic because it is linear and interpretable. If countries align along one dominant axis, this suggests that much of the variation may be explained by a general friction gradient. If countries form clusters or diverge along a second axis, that may indicate qualitatively distinct regulatory profiles.

PCA should be interpreted cautiously for ordinal categorical data. It is helpful for exploratory structure, but not definitive evidence of legal clusters. It is best read alongside MDS, UMAP, and hierarchical clustering.

In [ ]:
if HAS_SKLEARN:
    # Fill missing values with column medians, then global median if needed.
    X = numeric_no_self.copy().astype(float)
    X = X.apply(lambda col: col.fillna(col.median()), axis=0)
    X = X.fillna(np.nanmedian(X.values))

    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=2, random_state=7)
    coords = pca.fit_transform(X_scaled)

    pca_df = pd.DataFrame({
        "country": X.index,
        "PC1": coords[:, 0],
        "PC2": coords[:, 1],
    })

    fig, ax = plt.subplots(figsize=(12, 10), dpi=300)
    ax.scatter(pca_df["PC1"], pca_df["PC2"], s=45, color="#7D8FA6")

    for _, row in pca_df.iterrows():
        ax.text(row["PC1"], row["PC2"], row["country"], fontsize=7, ha="left", va="center")

    ax.axhline(0, color="lightgray", linewidth=0.8)
    ax.axvline(0, color="lightgray", linewidth=0.8)

    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)")
    ax.set_title("Country embedding based on outgoing legal compatibility profiles", fontsize=14, pad=14)

    fig.tight_layout()
    savefig(fig, "06_country_profile_pca_embedding")
    plt.show()


#### Interpretation of the PCA embedding

Countries that appear close together in PCA space have similar outgoing legal compatibility profiles. A long one-dimensional arrangement suggests a primary gradient from more permissive to more restrictive profiles. Distinct clusters would suggest qualitatively different governance regimes.

If the plot shows substantial label overlap, this should not automatically be treated as a failure. It may indicate that the data do not strongly support separation into many independent dimensions. The subsequent clustering, MDS, t-SNE, and UMAP sections test whether alternative representations provide clearer structure.

### 6.2 Clustered heatmap: reordering the matrix by profile similarity

The clustered heatmap takes the original matrix and reorders countries according to hierarchical clustering of their outgoing profiles. Unlike PCA, which projects countries into two dimensions, this approach preserves the matrix format while placing similar countries near one another.

The method computes a distance between every pair of country export profiles and then builds a hierarchical tree using average linkage. The resulting order is used to reorder both rows and columns. This can reveal governance blocs as contiguous blocks of similar colors.

### Interpretation and rationale: clustered directional heatmap

The clustered heatmap reorders countries according to similarity in their legal compatibility profiles. Unlike the original matrix, which follows the spreadsheet order, this plot places countries with similar outgoing profiles near one another. The goal is to reveal governance blocs or families of countries whose transfer relationships have similar structure.

This analysis is particularly useful because it preserves the matrix format while improving the visibility of latent organization. Blocks of similar color along rows or columns may indicate regulatory convergence, shared legal frameworks, or common patterns of interoperability. Conversely, isolated rows or columns may indicate countries with distinctive profiles.

The clustering is exploratory. It does not prove that countries belong to formal legal communities, but it helps generate hypotheses about grouping and regulatory similarity.

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

X = numeric_no_self.copy().astype(float)

# Fill missing values conservatively
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

# Cluster countries by outgoing profiles
row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = X.index[row_order]
X_clustered = numeric.loc[ordered_countries, ordered_countries]

fig, ax = plt.subplots(figsize=(22, 17), dpi=300)

im = ax.imshow(
    X_clustered.values.astype(float),
    cmap=CMAP,
    norm=NORM,
    interpolation="none",
    aspect="auto",
)

ax.set_xticks(range(len(ordered_countries)))
ax.set_yticks(range(len(ordered_countries)))
ax.set_xticklabels(ordered_countries, rotation=90, fontsize=7)
ax.set_yticklabels(ordered_countries, fontsize=7)

add_category_colorbar(fig, ax, im)

ax.set_title(
    "Clustered directional compatibility matrix",
    fontsize=15,
    pad=20,
)
ax.set_xlabel("Destination country / receiving jurisdiction", fontsize=11)
ax.set_ylabel("Origin country / exporting jurisdiction", fontsize=11)

fig.tight_layout()
savefig(fig, "06b_clustered_directional_heatmap")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

X = numeric_no_self.copy().astype(float)

# Fill missing values conservatively
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

# Cluster countries by outgoing profiles
row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = X.index[row_order]
X_clustered = numeric.loc[ordered_countries, ordered_countries]

# Square figure
fig, ax = plt.subplots(figsize=(24, 24), dpi=300)

im = ax.imshow(
    X_clustered.values.astype(float),
    cmap=CMAP,
    norm=NORM,
    interpolation="none",
    aspect="equal",  # keeps heatmap cells square
)

# Tick locations
ax.set_xticks(range(len(ordered_countries)))
ax.set_yticks(range(len(ordered_countries)))

# X-axis tick labels
ax.set_xticklabels(
    ordered_countries,
    rotation=90,
    ha="right",
    va="top",
    rotation_mode="anchor",
    fontsize=13,
)

# Y-axis tick labels
ax.set_yticklabels(
    ordered_countries,
    rotation=0,
    ha="right",
    va="center",
    fontsize=13,
)

# Padding
ax.tick_params(axis="x", pad=8)
ax.tick_params(axis="y", pad=6)


# ----------------------------
# Custom colorbar for this figure
# ----------------------------
legend_labels = [
'Directly enabled (same legal space)', 
 'Directly enabled (adequacy/ high convergence)', 
 'Permissible with standard transfer tools (DTA)', 
 'Permissible but legally conditional/ Friction heavy', 
 'Restricted/ Difficult', 
 'Legally uncertain/ Not presently supported'
]

cbar = fig.colorbar(
    im,
    ax=ax,
    ticks=range(len(legend_labels)),
    fraction=0.07,
    pad=0.05,
    shrink=0.50,
)

# Replace tick labels
cbar.set_ticks(range(len(legend_labels)))
cbar.set_ticklabels(legend_labels)

# Font size
cbar.ax.tick_params(labelsize=16)


# Legend title
cbar.set_label(
    "Directional Transfer Compatibility Class",
    fontsize=20,
    fontweight="bold",
    labelpad=20,
)

# Put title on left of colorbar
cbar.ax.yaxis.set_label_position("left")
cbar.ax.tick_params(labelsize=18)


# Title and axis labels
ax.set_title(
    "Clustered Directional Compatibility Matrix",
    fontsize=28,
    fontweight="bold",
    pad=25,
)

ax.set_xlabel(
    "Destination Country/ Receiving Jurisdiction",
    fontsize=22,
    fontweight="bold",
    labelpad=25,
)

ax.set_ylabel(
    "Origin Country/ Exporting Jurisdiction",
    fontsize=22,
    fontweight="bold",
    labelpad=25,
)

# Extra space for labels while keeping heatmap square
plt.subplots_adjust(
    bottom=0.28,
    left=0.24,
    right=0.90,
    top=0.92,
)

savefig(fig, "06b_clustered_directional_heatmap_square_pr")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Custom compatibility colors
# ----------------------------
LEVEL_COLORS = [
    "#2A9D8F",  # teal
    "#7FB069",  # sage
    "#E9C46A",  # sand
    "#F4A261",  # amber
    "#D16D6A",  # muted coral
    "#5C4B51",  # charcoal plum
]
CMAP = ListedColormap(LEVEL_COLORS)
NORM = BoundaryNorm(
    boundaries=np.arange(-0.5, len(LEVEL_COLORS) + 0.5, 1),
    ncolors=len(LEVEL_COLORS)
)

X = numeric_no_self.copy().astype(float)

# Fill missing values
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

# Cluster countries
row_dist = pdist(X.values, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

ordered_countries = X.index[row_order]
X_clustered = numeric.loc[ordered_countries, ordered_countries]

n = len(ordered_countries)
country_ids = np.arange(n)

# Softer categorical country colors
base = plt.cm.turbo(np.linspace(0, 1, n))

blend = 0.45   # blend toward white

country_colors = base.copy()
country_colors[:, :3] = (
    (1 - blend) * base[:, :3] +
    blend * 1.0
)

country_cmap = ListedColormap(country_colors)
fig, ax = plt.subplots(figsize=(24, 24), dpi=300)

# Main heatmap
im = ax.imshow(
    X_clustered.values.astype(float),
    cmap=CMAP,
    norm=NORM,
    interpolation="none",
    aspect="equal",
)

# Attach axes directly to matrix
divider = make_axes_locatable(ax)

ax_top = divider.append_axes(
    "top",
    size="3%",
    pad=0.03,
    sharex=ax
)

ax_left = divider.append_axes(
    "left",
    size="3%",
    pad=0.03,
    sharey=ax
)

# Top country color strip
ax_top.imshow(
    country_ids.reshape(1, n),
    cmap=country_cmap,
    interpolation="none",
    aspect="auto",
)

# Left country color strip
ax_left.imshow(
    country_ids.reshape(n, 1),
    cmap=country_cmap,
    interpolation="none",
    aspect="auto",
)

# Force exact alignment
ax.set_xlim(-0.5, n - 0.5)
ax.set_ylim(n - 0.5, -0.5)
ax_top.set_xlim(ax.get_xlim())
ax_left.set_ylim(ax.get_ylim())

# Remove ticks
ax.set_xticks([])
ax.set_yticks([])
ax_top.set_xticks([])
ax_top.set_yticks([])
ax_left.set_xticks([])
ax_left.set_yticks([])

# Country legend
legend_elements = [
    Patch(facecolor=country_colors[i], label=country)
    for i, country in enumerate(ordered_countries)
]

legend = ax.legend(
    handles=legend_elements,
    title="Countries",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    frameon=False,
    ncol=6,
    columnspacing=1.2,
    handlelength=1.2,
    handletextpad=0.5,
    prop={"size": 18},
    title_fontproperties={"size": 20, "weight": "bold"},
)
# Compatibility legend
legend_labels = [
    "Same legal space",
    "Adequacy decision",
    "Standard transfer tools",
    "Conditional transfer",
    "Restricted transfer",
    "Not supported",
]

cbar = fig.colorbar(
    im,
    ax=ax,
    ticks=range(len(legend_labels)),
    fraction=0.07,
    pad=0.05,
    shrink=0.50,
)

cbar.set_ticks(range(len(legend_labels)))
cbar.set_ticklabels(legend_labels)

cbar.ax.tick_params(labelsize=18)

cbar.set_label(
    "Transfer Compatibility",
    fontsize=20,
    fontweight="bold",
    labelpad=20,
)

cbar.ax.yaxis.set_label_position("left")

# Title and axis labels
ax.set_title(
    "Cross-Border Data Transfer Compatibility",
    fontsize=22,
    fontweight="bold",
    pad=40,
)

ax.set_xlabel(
    "Destination Country",
    fontsize=22,
    fontweight="bold",
    labelpad=15,
)

ax.set_ylabel(
    "Origin Country",
    fontsize=22,
    fontweight="bold",
    labelpad=40,
)

plt.subplots_adjust(
    bottom=0.28,
    left=0.24,
    right=0.90,
    top=0.90,
)

plt.rcParams["font.family"] = "Arial"

savefig(fig, "06b_clustered_directional_heatmap_country_soft_colors")
plt.show()

### 6.3 Geographic interpretation of clustered profile groups

The clustered heatmap reorders countries by similarity in their legal compatibility profiles, but the heatmap itself is not geographic. These maps ask whether the clusters revealed by hierarchical clustering have a spatial structure. If the same cluster appears in geographically coherent regions, this may suggest regional convergence, shared legal traditions, or common governance frameworks. If cluster membership is spatially dispersed, then the similarity may reflect legal alignment that is not purely geographic.

The first map shows discrete cluster membership. The second map shows the ordering of countries in the clustered heatmap. The order map is not a legal-friction measure; it is a diagnostic showing whether the matrix ordering has any geographic pattern. The number of clusters is adjustable and should be treated as an exploratory modeling choice.

In [ ]:
# Geographic maps of clustered country-profile groups
# This cell should be run after the clustered heatmap cell.

from scipy.cluster.hierarchy import fcluster

N_CLUSTERS = 4

cluster_labels = fcluster(
    row_linkage,
    t=N_CLUSTERS,
    criterion="maxclust",
)

cluster_df = pd.DataFrame({
    "country": X.index,
    "cluster": cluster_labels,
})

cluster_order_df = pd.DataFrame({
    "country": list(ordered_countries),
    "cluster_order": range(1, len(ordered_countries) + 1),
})

cluster_df = cluster_df.merge(cluster_order_df, on="country", how="left")
cluster_df["iso_a3"] = cluster_df["country"].map(country_to_iso3)

cluster_merged = world.merge(
    cluster_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)

def plot_cluster_membership_map(column, title, outfile_base):
    matched_geo = cluster_merged[cluster_merged[column].notna()].copy()

    print(f"\n{title}")
    print(f"Countries colored on map: {matched_geo['country'].nunique()}")

    if matched_geo.empty:
        print("No matched countries to plot.")
        return

    fig, ax = plt.subplots(figsize=(16, 9), dpi=300)

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    if column == "cluster":
        cmap = plt.get_cmap("tab10", int(matched_geo[column].max()))
        matched_geo.plot(
            column=column,
            ax=ax,
            cmap=cmap,
            categorical=True,
            legend=True,
            edgecolor="black",
            linewidth=0.45,
            zorder=2,
            legend_kwds={"title": "Profile cluster"},
        )
    else:
        matched_geo.plot(
            column=column,
            ax=ax,
            cmap="viridis",
            legend=True,
            edgecolor="black",
            linewidth=0.45,
            zorder=2,
            legend_kwds={"label": "Clustered matrix order"},
        )

    minx, miny, maxx, maxy = matched_geo.total_bounds
    ax.set_xlim(minx - (maxx - minx) * 0.05, maxx + (maxx - minx) * 0.05)
    ax.set_ylim(miny - (maxy - miny) * 0.08, maxy + (maxy - miny) * 0.08)

    ax.set_title(title, fontsize=15, pad=14)
    ax.axis("off")

    fig.tight_layout()
    savefig(fig, outfile_base)
    plt.show()


plot_cluster_membership_map(
    "cluster",
    f"Geographic distribution of {N_CLUSTERS} clustered legal-profile groups",
    "06b_world_map_cluster_membership",
)

plot_cluster_membership_map(
    "cluster_order",
    "Geographic distribution of country order in the clustered heatmap",
    "06c_world_map_cluster_order",
)


In [ ]:
print("Countries in matrix containing North American terms:")
print([
    c for c in df.index
    if any(s in c.lower() for s in ["canada", "united states", "america", "usa", "mexico"])
])

print("\nCountries in mode_df:")
print(
    mode_df[
        mode_df["country"].str.contains(
            "Canada|United States|America|USA|Mexico",
            case=False,
            na=False,
        )
    ][["country", "iso_a3", "import_mode", "export_mode"]]
)

#### Interpretation of the clustered heatmap

Blocks along the diagonal indicate groups of countries that are mutually similar in their legal compatibility profiles. Off-diagonal blocks reveal how one cluster relates to another. If the clustered heatmap reveals sharper visual structure than the original matrix, it suggests that legal interoperability is organized by latent governance clusters rather than by the original country order.

This figure is often more interpretable than a PCA scatterplot for categorical matrices because it preserves the full relationship structure while improving the ordering of rows and columns.

### 6.3 MDS: distance-preserving country embedding

Multidimensional Scaling (MDS) represents countries in two dimensions while attempting to preserve the pairwise distances between their legal profiles. Whereas PCA maximizes explained variance along linear axes, MDS begins from a distance matrix and tries to maintain similarity relationships directly.

This makes MDS attractive for this dataset because the substantive question is relational: which countries have similar profiles of legal compatibility toward the rest of the world?

### Interpretation and rationale: multidimensional scaling embedding

Multidimensional scaling, or MDS, represents countries in a low-dimensional space while attempting to preserve pairwise distances between their legal profiles. Countries placed near each other have similar outgoing compatibility profiles; countries far apart have more different profiles.

MDS is often more appropriate than PCA when the main object of interest is pairwise similarity rather than variance along linear axes. In this notebook, each country’s outgoing legal profile is converted into a distance matrix, and MDS finds a two-dimensional configuration that approximates those distances.

This plot should be interpreted as a similarity map of country profiles, not as a geographic map. Axes have no intrinsic legal meaning; distances among points are the meaningful feature.

In [ ]:
from sklearn.manifold import MDS
from scipy.spatial.distance import pdist, squareform

X = numeric_no_self.copy().astype(float)

# Fill missing values
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

# Compute pairwise distances between countries based on outgoing profiles
distance_matrix = squareform(pdist(X.values, metric="euclidean"))

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=7,
    normalized_stress="auto",
)

coords = mds.fit_transform(distance_matrix)

mds_df = pd.DataFrame({
    "country": X.index,
    "MDS1": coords[:, 0],
    "MDS2": coords[:, 1],
})

fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

ax.scatter(
    mds_df["MDS1"],
    mds_df["MDS2"],
    s=45,
    color="#7D8FA6",
)

for _, row in mds_df.iterrows():
    ax.text(
        row["MDS1"],
        row["MDS2"],
        row["country"],
        fontsize=7,
        ha="left",
        va="center",
    )

ax.axhline(0, color="lightgray", linewidth=0.8)
ax.axvline(0, color="lightgray", linewidth=0.8)

ax.set_xlabel("MDS dimension 1")
ax.set_ylabel("MDS dimension 2")
ax.set_title(
    "MDS embedding of countries based on outgoing legal compatibility profiles",
    fontsize=14,
    pad=14,
)

fig.tight_layout()
savefig(fig, "06c_country_profile_mds_embedding")
plt.show()

#### Interpretation of the MDS embedding

Countries close together in the MDS plot have similar outgoing legal-friction profiles. Distances between points are more directly meaningful than in t-SNE or UMAP because MDS explicitly attempts to preserve the original pairwise distance matrix.

If MDS separates countries more clearly than PCA, it suggests that the structure is not well summarized by linear variance axes alone. If MDS also collapses into a gradient, the data likely contain a dominant one-dimensional legal-friction continuum.

### 6.4 t-SNE: local-neighborhood exploratory embedding

t-SNE is a nonlinear visualization method designed to preserve local neighborhoods. It can be useful for detecting small groups of countries with very similar profiles, but it does not preserve global distances reliably. Two clusters that appear far apart in t-SNE space are not necessarily proportionally far apart in the original data.

For this reason, t-SNE should be treated as an exploratory diagnostic rather than a primary inferential figure. Its value is mainly in asking whether local neighborhoods appear that were not visible in PCA or MDS.

### Interpretation and rationale: t-SNE embedding

t-SNE is a nonlinear embedding method designed to preserve local neighborhoods. In this analysis, countries with similar legal profiles may be placed close together even when global distances are distorted. It can reveal local clusters that linear methods such as PCA may obscure.

Because t-SNE is stochastic and sensitive to parameters such as perplexity, it should be treated as exploratory rather than confirmatory. The absolute position of points and the distance between distant clusters are not generally interpretable. What matters most is whether nearby countries form stable, policy-interpretable neighborhoods.

This plot is useful as a diagnostic for local structure but should not be treated as the primary evidence for governance clusters unless its structure is robust across parameter settings.

In [ ]:
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform

X = numeric_no_self.copy().astype(float)

# Fill missing values
X = X.apply(lambda col: col.fillna(col.median()), axis=0)
X = X.fillna(np.nanmedian(X.values))

# Pairwise distances between countries based on outgoing profiles
distance_matrix = squareform(pdist(X.values, metric="euclidean"))

# t-SNE is sensitive to sample size. Perplexity must be less than n_samples.
n_samples = X.shape[0]
perplexity = min(30, max(5, (n_samples - 1) // 3))

tsne = TSNE(
    n_components=2,
    metric="precomputed",
    perplexity=perplexity,
    learning_rate="auto",
    init="random",
    random_state=7,
)

coords = tsne.fit_transform(distance_matrix)

tsne_df = pd.DataFrame({
    "country": X.index,
    "TSNE1": coords[:, 0],
    "TSNE2": coords[:, 1],
})

fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

ax.scatter(
    tsne_df["TSNE1"],
    tsne_df["TSNE2"],
    s=45,
    color="#7D8FA6",
)

for _, row in tsne_df.iterrows():
    ax.text(
        row["TSNE1"],
        row["TSNE2"],
        row["country"],
        fontsize=7,
        ha="left",
        va="center",
    )

ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.set_title(
    f"t-SNE embedding of countries based on outgoing legal compatibility profiles\n"
    f"perplexity = {perplexity}",
    fontsize=14,
    pad=14,
)

fig.tight_layout()
savefig(fig, "06d_country_profile_tsne_embedding")
plt.show()

#### Interpretation of the t-SNE embedding

Interpret t-SNE by focusing on local neighborhoods: countries plotted close together are likely to have similar outgoing legal profiles. Avoid overinterpreting the distance between separated clusters or the overall orientation of the plot. Because t-SNE is sensitive to perplexity and random initialization, stable findings should be checked by rerunning with different seeds or perplexity values.

If t-SNE reveals groups that also appear in the clustered heatmap or MDS plot, those groups are more likely to reflect robust structure in the data.

### 6.5 UMAP: nonlinear embedding balancing local and global structure

UMAP is another nonlinear embedding method. Compared with t-SNE, it often preserves more of the broader global structure while still emphasizing local neighborhoods. The `n_neighbors` parameter controls the scale of structure emphasized: smaller values emphasize local neighborhoods, while larger values emphasize broader organization.

UMAP can be visually effective, but it is still an exploratory representation. It should be used to generate hypotheses about governance clusters, not to prove them.

### Interpretation and rationale: UMAP embedding

UMAP is another nonlinear embedding method that attempts to preserve both local and some intermediate-scale structure. Compared with t-SNE, UMAP often produces more stable global organization, although it remains parameter-dependent. In this notebook, UMAP is used to explore whether country legal profiles form clusters or gradients that are not visible in PCA.

As with all embeddings, the axes do not have direct substantive interpretation. The meaningful information lies in the relative arrangement of countries. Countries near each other have similar outgoing compatibility profiles under the chosen distance metric.

UMAP is useful for exploratory visualization and may be helpful for identifying candidate governance clusters, but any apparent cluster should be checked against the matrix, stacked profiles, and clustered heatmap.

In [ ]:
# If needed, install UMAP first:
# import sys
!pip install umap-learn

from scipy.spatial.distance import pdist, squareform

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print(
        "UMAP is not installed. Install it with: "
        "pip install umap-learn"
    )

if HAS_UMAP:
    X = numeric_no_self.copy().astype(float)

    # Fill missing values
    X = X.apply(lambda col: col.fillna(col.median()), axis=0)
    X = X.fillna(np.nanmedian(X.values))

    # Pairwise distances between countries based on outgoing profiles
    distance_matrix = squareform(pdist(X.values, metric="euclidean"))

    n_samples = X.shape[0]
    n_neighbors = min(15, max(3, n_samples // 5))

    reducer = umap.UMAP(
        n_components=2,
        metric="precomputed",
        n_neighbors=n_neighbors,
        min_dist=0.15,
        random_state=7,
    )

    coords = reducer.fit_transform(distance_matrix)

    umap_df = pd.DataFrame({
        "country": X.index,
        "UMAP1": coords[:, 0],
        "UMAP2": coords[:, 1],
    })

    fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

    ax.scatter(
        umap_df["UMAP1"],
        umap_df["UMAP2"],
        s=45,
        color="#7D8FA6",
    )

    for _, row in umap_df.iterrows():
        ax.text(
            row["UMAP1"],
            row["UMAP2"],
            row["country"],
            fontsize=7,
            ha="left",
            va="center",
        )

    ax.set_xlabel("UMAP dimension 1")
    ax.set_ylabel("UMAP dimension 2")
    ax.set_title(
        f"UMAP embedding of countries based on outgoing legal compatibility profiles\n"
        f"n_neighbors = {n_neighbors}, min_dist = 0.15",
        fontsize=14,
        pad=14,
    )

    fig.tight_layout()
    savefig(fig, "06e_country_profile_umap_embedding")
    plt.show()

#### Interpretation of the UMAP embedding

Countries close together in UMAP space have similar outgoing legal compatibility profiles under the chosen distance metric and parameter settings. If UMAP reveals clear clusters that correspond to known legal or regional groupings, this may indicate meaningful governance structure. If the plot is unstable across parameters, the apparent clusters should be treated cautiously.

For manuscript purposes, UMAP is strongest when used alongside the clustered heatmap. The heatmap provides a transparent relationship-level view, while UMAP provides an accessible two-dimensional summary.

## 7. Entropy / diversity profile and map

The preceding embeddings ask whether countries have similar profiles. This section asks a complementary question: **how heterogeneous is each country’s profile?** Entropy summarizes the diversity of legal classes within each country’s export or import relationships.

A country with low entropy has a relatively uniform profile: most transfers fall into the same class. A country with high entropy has a mixed profile: different destination or origin countries fall into different legal categories. Entropy therefore captures regulatory heterogeneity rather than permissiveness alone.

This is important because two countries can have the same modal class but very different distributions. Entropy helps distinguish a stable legal profile from a fragmented one.

### Interpretation and rationale: exploratory plot

This cell generates a plot or saved visual output that contributes to the exploratory analysis of the cross-border legal interoperability matrix. Interpret the figure in relation to the preceding analyses: each plot either preserves, summarizes, aggregates, or re-expresses the directed country-by-country matrix. When interpreting the result, consider how much information has been compressed from the original matrix and whether the plot is best suited for pairwise, country-level, regional, or global inference.

In [ ]:
def shannon_entropy(categories):
    values = pd.Series(categories).dropna()
    if values.empty:
        return np.nan
    p = values.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())


entropy_df = pd.DataFrame({
    "country": df_no_self.index,
    "export_entropy": df_no_self.apply(lambda row: shannon_entropy(row.values), axis=1).values,
    "import_entropy": df_no_self.apply(lambda col: shannon_entropy(col.values), axis=0).reindex(df_no_self.index).values,
})

entropy_sorted = entropy_df.sort_values("export_entropy", ascending=False)

fig, ax = plt.subplots(figsize=(10, max(8, len(entropy_sorted) * 0.35)), dpi=300)
ax.barh(entropy_sorted["country"], entropy_sorted["export_entropy"], color="#7D8FA6")
ax.invert_yaxis()
ax.set_xlabel("Shannon entropy of outgoing compatibility profile")
ax.set_ylabel("Country / origin jurisdiction")
ax.set_title("Heterogeneity of country export compatibility profiles", fontsize=14, pad=14)

fig.tight_layout()
savefig(fig, "07a_export_entropy_barplot")
plt.show()


### Interpretation and rationale: entropy maps

The entropy maps quantify heterogeneity in each country’s legal profile. Entropy is high when a country’s relationships are spread across several compatibility classes and low when most relationships fall into one class. In this context, entropy does not indicate better or worse interoperability; it indicates diversity or inconsistency of legal classifications across partners.

These maps complement both the mode and mean-friction maps. A country can have moderate mean friction and still have either low entropy, if most relationships are consistently C, or high entropy, if it has a mixture of A, B, C, D, and E relationships. Thus, entropy helps distinguish uniform regulatory profiles from heterogeneous ones.

Interpretively, high entropy may indicate complex or partner-dependent governance conditions, whereas low entropy may indicate a more uniform legal regime. These maps are particularly useful for identifying countries whose single-color modal summary is potentially misleading.

In [ ]:
# Entropy maps
# Entropy quantifies heterogeneity of a country's import/export profile.

def shannon_entropy(values):
    values = pd.Series(values).dropna()

    if values.empty:
        return np.nan

    p = values.value_counts(normalize=True)

    return float(-(p * np.log2(p)).sum())


entropy_map_df = pd.DataFrame({
    "country": df_no_self.index,

    "export_entropy":
        df_no_self.apply(
            lambda row: shannon_entropy(row.values),
            axis=1
        ).values,

    "import_entropy":
        df_no_self.apply(
            lambda col: shannon_entropy(col.values),
            axis=0
        ).reindex(df_no_self.index).values,
})

entropy_map_df["iso_a3"] = (
    entropy_map_df["country"]
    .map(country_to_iso3)
)

entropy_merged = world.merge(
    entropy_map_df,
    left_on="ne_iso_a3",
    right_on="iso_a3",
    how="left",
)


def plot_entropy_map(
    column,
    title,
    outfile_base,
):

    matched_geo = entropy_merged[
        entropy_merged[column].notna()
    ].copy()

    print(f"\n{title}")
    print(
        f"Countries colored on map: "
        f"{matched_geo['country'].nunique()}"
    )

    if matched_geo.empty:
        print("No matched countries to plot.")
        return

    fig, ax = plt.subplots(
        figsize=(16, 9),
        dpi=300,
    )

    world.plot(
        ax=ax,
        color="#F2F2F2",
        edgecolor="#D0D0D0",
        linewidth=0.25,
        zorder=1,
    )

    matched_geo.plot(
        column=column,
        ax=ax,
        cmap="cividis",
        legend=True,
        edgecolor="black",
        linewidth=0.45,
        zorder=2,
        legend_kwds={
            "label": "Shannon entropy"
        },
    )

    minx, miny, maxx, maxy = matched_geo.total_bounds

    ax.set_xlim(
        minx - (maxx - minx) * 0.05,
        maxx + (maxx - minx) * 0.05,
    )

    ax.set_ylim(
        miny - (maxy - miny) * 0.08,
        maxy + (maxy - miny) * 0.08,
    )

    ax.set_title(
        title,
        fontsize=15,
        pad=14,
    )

    ax.axis("off")

    fig.tight_layout()

    savefig(fig, outfile_base)

    plt.show()


plot_entropy_map(
    "import_entropy",
    "Import-profile entropy by receiving jurisdiction",
    "07g_world_map_import_entropy",
)

plot_entropy_map(
    "export_entropy",
    "Export-profile entropy by exporting jurisdiction",
    "07h_world_map_export_entropy",
)

### Interpretation of entropy analyses

High export entropy indicates that a country’s outgoing transfer conditions vary substantially across destinations. Low export entropy indicates a more uniform export regime. High import entropy indicates that other countries face varied conditions when sending data to that country, whereas low import entropy indicates a more consistent receiving profile.

Entropy does not indicate whether a country is permissive or restrictive. It indicates whether its profile is homogeneous or heterogeneous. Therefore, entropy should be interpreted together with mean friction, mode, and stacked proportions.

## 8. Sankey / alluvial-style summary

The previous section measured heterogeneity country by country. This section aggregates across countries to examine how country-level dominant profiles relate to individual transfer-level classes. Each directed transfer is classified by the export-mode class of its origin country and by its actual bilateral transfer class.

The Sankey or alluvial representation shows whether countries with a given dominant profile mostly generate transfers of the same class, or whether their individual bilateral relationships spread across multiple classes. This is a bridge between country-level summaries and relationship-level data.

### Interpretation and rationale: Sankey or alluvial summary

The Sankey-style summary aggregates many directed relationships into flows between country-level profile classes and transfer-level outcomes. It is designed to show how dominant country profiles translate into the distribution of individual transfer classes.

This representation is useful for communicating the global composition of the matrix in a compact way. It helps show whether countries with a given modal export profile mostly generate the same class of transfer relationships or whether their individual transfer outcomes are more heterogeneous.

The Sankey plot is a summary, not a pairwise analysis. It is best used to describe aggregate structure and to complement more detailed country-level and pairwise figures.

In [ ]:
# Prepare transfer-level summary.
records = []
for origin in df_no_self.index:
    origin_mode = export_mode.loc[origin]
    for dest in df_no_self.columns:
        cls = df_no_self.loc[origin, dest]
        if pd.notna(cls):
            records.append({
                "origin": origin,
                "destination": dest,
                "origin_export_mode": origin_mode,
                "transfer_class": cls,
            })

flow_df = pd.DataFrame(records)
flow_counts = (
    flow_df
    .groupby(["origin_export_mode", "transfer_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0)
)

display(flow_counts)


### Interpretation and rationale: Sankey or alluvial summary

The Sankey-style summary aggregates many directed relationships into flows between country-level profile classes and transfer-level outcomes. It is designed to show how dominant country profiles translate into the distribution of individual transfer classes.

This representation is useful for communicating the global composition of the matrix in a compact way. It helps show whether countries with a given modal export profile mostly generate the same class of transfer relationships or whether their individual transfer outcomes are more heterogeneous.

The Sankey plot is a summary, not a pairwise analysis. It is best used to describe aggregate structure and to complement more detailed country-level and pairwise figures.

In [ ]:
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly is not installed. Static heatmap fallback will be used.")

if HAS_PLOTLY:
    left_nodes = [f"Origin mode {c}" for c in CATEGORY_ORDER]
    right_nodes = [f"Transfer {c}" for c in CATEGORY_ORDER]
    labels = left_nodes + right_nodes

    sources, targets, values, colors = [], [], [], []
    for i, origin_class in enumerate(CATEGORY_ORDER):
        for j, transfer_class in enumerate(CATEGORY_ORDER):
            val = int(flow_counts.loc[origin_class, transfer_class])
            if val > 0:
                sources.append(i)
                targets.append(len(CATEGORY_ORDER) + j)
                values.append(val)
                colors.append(PALETTE[transfer_class])

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=16,
            thickness=18,
            line=dict(color="black", width=0.3),
            label=labels,
            color=[PALETTE[c] for c in CATEGORY_ORDER] + [PALETTE[c] for c in CATEGORY_ORDER],
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors,
        )
    )])

    fig.update_layout(title_text="Alluvial summary of origin export mode to transfer-level class", font_size=11)
    fig.write_html(OUTDIR / "08_sankey_origin_mode_to_transfer_class.html")
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    im = ax.imshow(flow_counts.values, cmap=CMAP, aspect="auto")
    ax.set_xticks(range(len(CATEGORY_ORDER)))
    ax.set_yticks(range(len(CATEGORY_ORDER)))
    ax.set_xticklabels(CATEGORY_ORDER)
    ax.set_yticklabels(CATEGORY_ORDER)
    ax.set_xlabel("Transfer-level class")
    ax.set_ylabel("Origin export-mode class")
    ax.set_title("Static flow-summary heatmap")

    for i in range(flow_counts.shape[0]):
        for j in range(flow_counts.shape[1]):
            ax.text(j, i, int(flow_counts.iloc[i, j]), ha="center", va="center", fontsize=8)

    fig.tight_layout()
    savefig(fig, "08_static_flow_summary_heatmap")
    plt.show()


### Interpretation of the Sankey / flow summary

Wide flows indicate common combinations of origin export-mode class and actual transfer class. If most flow remains on the diagonal, country-level modal profiles are good summaries of the underlying transfer relationships. If flows spread widely across classes, then the mode is a poor summary and country profiles are more heterogeneous.

This plot is especially useful for evaluating whether modal maps are adequate. Strong off-diagonal flows indicate that the maps may be visually appealing but analytically incomplete.

## 9. Region-aggregated block matrix

The preceding analyses operate at the country level. This section aggregates the directed matrix into broader regions. The purpose is to ask whether legal friction is organized at the macro-regional scale, for example within Europe/UK, within Africa, or between Africa and Europe/UK.

The default region mapping is intentionally editable. It should be aligned with the regions used in the manuscript and with the coding logic used to construct the original matrix. The block matrix reports the mean ordinal friction across all directed country pairs from each origin region to each destination region.

### Interpretation and rationale: mean legal-friction maps

The mean-friction maps provide a different type of country-level summary. Instead of using the most frequent class, they convert the ordered legal categories into an ordinal friction scale, with A1 as lowest friction and E as highest friction, and then compute the average incoming or outgoing friction for each country.

These maps are useful when the question concerns the overall severity of legal difficulty rather than the dominant category. A country may have a modal class of C, but its mean friction can distinguish whether its non-modal relationships are mostly easier, such as B, or harder, such as D and E. The mean-friction maps therefore capture information that the mode maps discard.

Because the legal classes are ordinal rather than truly continuous, the mean should be interpreted as a pragmatic summary index rather than a precise metric quantity. It is useful for comparing broad levels of relative friction, especially when paired with the full matrix and stacked profiles.

In [ ]:
# Edit or extend this dictionary to match your preferred regional classification.
EUROPE_UK = {
    "Germany","France","Italy","Spain","Netherlands","Belgium","Sweden","Austria","Finland",
    "Ireland","Denmark","Luxembourg","Czechia","Portugal","Lithuania","Estonia","Poland",
    "Greece","Hungary","Slovakia","Slovenia","Romania","United Kingdom","Switzerland",
    "Norway","Iceland","Croatia","Bulgaria","Latvia","Malta","Cyprus"
}

NORTH_AMERICA = {"United States", "United States of America", "Canada", "Mexico"}
LATIN_AMERICA = {
    "Argentina","Brazil","Chile","Colombia","Costa Rica","Peru","Uruguay","Ecuador",
    "Bolivia","Paraguay","Venezuela","Panama","Guatemala","Honduras","Nicaragua",
    "El Salvador","Dominican Republic"
}

def infer_region(country):
    if country in EUROPE_UK:
        return "Europe/UK"
    if country in NORTH_AMERICA:
        return "North America"
    if country in LATIN_AMERICA:
        return "Latin America"
    return "Africa / other"

region_map = {country: infer_region(country) for country in df_no_self.index}
regions = sorted(set(region_map.values()))

region_numeric = numeric_no_self.copy()
region_records = []
for origin_region in regions:
    origin_countries = [c for c in region_numeric.index if region_map[c] == origin_region]
    for dest_region in regions:
        dest_countries = [c for c in region_numeric.columns if region_map[c] == dest_region]
        vals = region_numeric.loc[origin_countries, dest_countries].to_numpy(copy=True).flatten()
        vals = pd.Series(vals).dropna().astype(float).values
        mean_val = np.nan if len(vals) == 0 else float(np.mean(vals))
        mode_val = np.nan if len(vals) == 0 else int(pd.Series(vals).mode().iloc[0])
        region_records.append({
            "origin_region": origin_region,
            "destination_region": dest_region,
            "mean_friction": mean_val,
            "mode_value": mode_val,
            "mode_class": VALUE_TO_CATEGORY.get(mode_val, np.nan),
        })

region_df = pd.DataFrame(region_records)
display(region_df)


### Interpretation and rationale: mean legal-friction maps

The mean-friction maps provide a different type of country-level summary. Instead of using the most frequent class, they convert the ordered legal categories into an ordinal friction scale, with A1 as lowest friction and E as highest friction, and then compute the average incoming or outgoing friction for each country.

These maps are useful when the question concerns the overall severity of legal difficulty rather than the dominant category. A country may have a modal class of C, but its mean friction can distinguish whether its non-modal relationships are mostly easier, such as B, or harder, such as D and E. The mean-friction maps therefore capture information that the mode maps discard.

Because the legal classes are ordinal rather than truly continuous, the mean should be interpreted as a pragmatic summary index rather than a precise metric quantity. It is useful for comparing broad levels of relative friction, especially when paired with the full matrix and stacked profiles.

In [ ]:
block = region_df.pivot(index="origin_region", columns="destination_region", values="mean_friction").loc[regions, regions]

fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
im = ax.imshow(block.values.astype(float), cmap=CMAP, norm=NORM, interpolation="none", aspect="equal")

ax.set_xticks(range(len(regions)))
ax.set_yticks(range(len(regions)))
ax.set_xticklabels(regions, rotation=45, ha="right")
ax.set_yticklabels(regions)

for i in range(len(regions)):
    for j in range(len(regions)):
        val = block.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)

add_category_colorbar(fig, ax, im, label="Mean legal friction across directed country pairs")
ax.set_title("Region-aggregated mean transfer friction", fontsize=14, pad=14)
ax.set_xlabel("Destination region")
ax.set_ylabel("Origin region")

fig.tight_layout()
savefig(fig, "09_region_aggregated_block_matrix")
plt.show()


### Interpretation of the region block matrix

Each cell summarizes the average legal friction for transfers from one region to another. Low values indicate broadly enabled or feasible data sharing. High values indicate greater conditionality, restriction, or uncertainty. Asymmetry between two off-diagonal cells, such as Region A → Region B versus Region B → Region A, suggests direction-specific regional imbalance.

Because this plot averages across many country pairs, it can hide within-region heterogeneity. It should be interpreted as a macro-level summary and checked against the country-level matrix and stacked profiles.

## 10. Parallel-coordinate country profiles

The stacked bar plot showed each country’s outgoing profile as a horizontal composition. This section uses a parallel-coordinate representation of the same information. Each line corresponds to one country, and the six axes correspond to the proportions of A1, A2, B, C, D, and E outgoing transfers.

This plot emphasizes the shape of profiles rather than country labels. It is useful for identifying whether countries follow a common pattern, whether there are multiple profile types, and whether some legal classes dominate the distribution.

### Interpretation and rationale: parallel-coordinate country profiles

The parallel-coordinate plot represents each country as a line across the six legal classes. The y-value at each class is the proportion of that country’s outgoing relationships in that class. The median profile is overlaid as a reference.

This plot is useful for seeing whether country profiles share a common shape or whether some countries deviate strongly from the typical pattern. It provides a compact view of distributional heterogeneity across all countries, although individual country labels are usually not shown because the purpose is to visualize profile shapes.

This analysis complements the stacked bar plot. The stacked plot emphasizes country-by-country readability; the parallel-coordinate plot emphasizes the ensemble of profile shapes.

In [ ]:
parallel_df = outgoing_dist.copy()
parallel_df["country"] = parallel_df.index

fig, ax = plt.subplots(figsize=(10, 7), dpi=300)

x = np.arange(len(CATEGORY_ORDER))
for _, row in parallel_df.iterrows():
    y = [row[c] for c in CATEGORY_ORDER]
    ax.plot(x, y, color="#7D8FA6", alpha=0.35, linewidth=1)

# Overlay median profile.
median_profile = parallel_df[CATEGORY_ORDER].median()
ax.plot(x, median_profile.values, color="black", linewidth=2.5, label="Median country profile")

ax.set_xticks(x)
ax.set_xticklabels(CATEGORY_ORDER)
ax.set_ylabel("Proportion of outgoing transfers")
ax.set_xlabel("Compatibility class")
ax.set_title("Parallel-coordinate view of outgoing country profiles", fontsize=14, pad=14)
ax.legend(frameon=False)

fig.tight_layout()
savefig(fig, "10_parallel_coordinate_country_profiles")
plt.show()


### Interpretation of the parallel-coordinate plot

The black line shows the median country profile. Individual gray lines show how each country deviates from that median. If most lines follow a similar shape, the countries share a common legal-profile structure. If lines cross heavily or spread widely, country profiles are heterogeneous.

This plot is less useful for identifying specific countries, but it is valuable for seeing whether the dataset contains common profile archetypes.

## 11. Row-normalized country-by-class heatmap

This section presents a compact alternative to the stacked bar plot. Each row is a country, each column is a legal class, and each cell is the proportion of that country’s outgoing transfers falling into that class.

The row-normalized heatmap preserves the same information as the stacked bar chart but emphasizes comparative intensity across classes. It is often easier to scan when the number of countries is large.

### Interpretation and rationale: row-normalized country-by-class heatmap

The row-normalized heatmap is a compact alternative to the stacked bar plot. Each row is a country, each column is a legal class, and each cell gives the proportion of that country’s outgoing relationships in that class. This makes it easier to compare countries across classes without the visual density of stacked bars.

This plot is useful when the main question is which classes dominate each country’s outgoing profile. Strong vertical bands indicate classes that are common across many countries. Strong row-specific patterns indicate countries with distinctive distributions.

It should be interpreted as a distributional summary rather than a pairwise matrix. It tells us what kinds of relationships a country tends to have, not which specific partners produce them.

In [ ]:
heat = outgoing_dist[CATEGORY_ORDER].copy()

fig, ax = plt.subplots(figsize=(8, max(10, len(heat) * 0.35)), dpi=300)
im = ax.imshow(heat.values, cmap="Blues", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(CATEGORY_ORDER)))
ax.set_xticklabels(CATEGORY_ORDER)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Proportion of outgoing transfers", fontsize=10)

ax.set_xlabel("Compatibility class")
ax.set_ylabel("Origin jurisdiction")
ax.set_title("Row-normalized outgoing compatibility profile heatmap", fontsize=14, pad=14)

fig.tight_layout()
savefig(fig, "11_row_normalized_country_class_heatmap")
plt.show()


### Interpretation of the row-normalized heatmap

Dark cells indicate that a large proportion of a country’s outgoing transfers fall into that class. A country with one dark cell and several light cells has a concentrated profile. A country with multiple medium-intensity cells has a more heterogeneous profile.

This plot provides a useful bridge between stacked profiles and entropy: visually concentrated rows should correspond to lower entropy, while visually diffuse rows should correspond to higher entropy.

## 12. Inequality / Lorenz-style analysis of average legal friction

The preceding sections describe structure, clustering, and heterogeneity. This final quantitative section asks whether legal friction is evenly distributed across countries or concentrated among a subset of jurisdictions.

For each origin country, the notebook computes the average outgoing legal-friction score using A1 = 0 through E = 5. The ranked bar plot shows which countries have higher or lower average friction. The Lorenz-style curve and Gini coefficient summarize whether friction is concentrated unevenly across countries.

### Interpretation and rationale: exploratory plot

This cell generates a plot or saved visual output that contributes to the exploratory analysis of the cross-border legal interoperability matrix. Interpret the figure in relation to the preceding analyses: each plot either preserves, summarizes, aggregates, or re-expresses the directed country-by-country matrix. When interpreting the result, consider how much information has been compressed from the original matrix and whether the plot is best suited for pairwise, country-level, regional, or global inference.

In [ ]:
avg_export_friction = numeric_no_self.mean(axis=1, skipna=True).sort_values()
friction_df = pd.DataFrame({
    "country": avg_export_friction.index,
    "average_export_friction": avg_export_friction.values,
})

display(friction_df)

fig, ax = plt.subplots(figsize=(10, max(8, len(friction_df) * 0.3)), dpi=300)
ax.barh(friction_df["country"], friction_df["average_export_friction"], color="#7D8FA6")
ax.set_xlabel("Average outgoing legal friction (A1 = 0, E = 5)")
ax.set_ylabel("Country / origin jurisdiction")
ax.set_title("Average outgoing legal friction by country", fontsize=14, pad=14)

fig.tight_layout()
savefig(fig, "12a_average_export_friction_by_country")
plt.show()


### Interpretation and rationale: inequality and Lorenz-style friction analysis

The inequality analysis treats each country’s average outgoing legal friction as a country-level burden or cost. The ranked bar plot shows which countries have lower or higher average export friction. The Lorenz-style curve then asks whether friction is evenly distributed across countries or concentrated among a subset.

This analysis is useful for moving from visualization to quantification. A high degree of inequality would suggest that legal friction is not evenly distributed across the global system, but instead disproportionately affects some jurisdictions. A lower inequality value would suggest a more uniform distribution of friction.

Because legal classes are ordinal, the friction score should be interpreted as an index rather than a precise continuous measurement. Nonetheless, it provides a useful way to summarize global imbalance.

In [ ]:
def gini_coefficient(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0 or np.all(x == 0):
        return np.nan
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

x = avg_export_friction.dropna().values
x_sorted = np.sort(x)
cum_friction = np.cumsum(x_sorted)
cum_friction_share = cum_friction / cum_friction[-1]
cum_country_share = np.arange(1, len(x_sorted) + 1) / len(x_sorted)

gini = gini_coefficient(x_sorted)

fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Equality line")
ax.plot(np.r_[0, cum_country_share], np.r_[0, cum_friction_share], color="#7D8FA6", linewidth=2.5)

ax.set_xlabel("Cumulative share of countries")
ax.set_ylabel("Cumulative share of average legal friction")
ax.set_title(f"Lorenz-style curve for outgoing legal friction\nGini coefficient = {gini:.3f}", fontsize=14, pad=14)
ax.legend(frameon=False)

fig.tight_layout()
savefig(fig, "12b_lorenz_curve_average_export_friction")
plt.show()


### Interpretation of the inequality analysis

The average-friction bar plot ranks countries by their typical outgoing legal difficulty. Higher values indicate more restrictive, conditional, or uncertain export environments. The Lorenz-style curve summarizes concentration: a curve close to the equality line indicates that friction is relatively evenly distributed across countries. A curve far below the equality line indicates that a smaller subset of countries accounts for a disproportionate share of legal friction.

The Gini coefficient is a compact scalar summary of this inequality. It should not replace the country-level plot, but it provides a useful way to compare versions of the matrix, regions, or future recoding iterations.

## Integrated interpretation guide

The notebook is designed as a sequence of increasingly abstract views of the same directed legal-interoperability matrix. The directional matrix is the primary evidence because it preserves all bilateral relationships. The stacked profiles, modal maps, entropy analyses, and average-friction measures summarize country-level behavior. The asymmetry matrix isolates reciprocity and nonreciprocity. The network analysis identifies hubs and bridges among feasible transfer pathways. The embedding and clustering analyses search for latent governance structure. The region block matrix provides a macro-level summary, and the inequality analysis quantifies whether friction is concentrated or broadly distributed.

A defensible manuscript interpretation should not rely on a single plot. Strong conclusions are those that appear consistently across multiple representations. For example, a country that appears restrictive in the matrix, has a high C/D/E stacked profile, high average friction, and low network out-strength is robustly characterized as a difficult exporter. A cluster that appears in the clustered heatmap, MDS, and UMAP is more persuasive than a cluster that appears only in t-SNE.

The recommended figure hierarchy is:

1. Use the directional matrix as the primary descriptive figure.
2. Use the asymmetry matrix to quantify reciprocity and directionality.
3. Use stacked profiles or row-normalized heatmaps to summarize country-level distributions.
4. Use clustering or MDS to identify governance families.
5. Use maps and networks as communicative secondary or supplementary figures.

## Technical note: read-only array fix

Some pandas and NumPy configurations expose `DataFrame.values` as a read-only view. Directly applying `np.fill_diagonal()` to such an array can produce the error:

```python
ValueError: underlying array is read-only
```

This notebook avoids that issue by using `remove_self_transfers()`, which first creates a writable NumPy copy and then replaces the diagonal with `NaN`. This approach is safer and should be used whenever self-transfers need to be removed from the matrix.